<a href="https://colab.research.google.com/github/epkewa/fine-tuned-kazakh-model-for-grammar/blob/main/kazakh_small_model_to_correct_grammar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install -U datasets huggingface_hub pyarrow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.9 MB/s eta 0:00:00


In [ ]:
import os
from pathlib import Path

# True — сохранять результат в Google Drive.
# Проверяющий может поставить False и сохранить всё временно в Colab.
USE_GOOGLE_DRIVE = True

# False — окончательные размеры.
# True — небольшой быстрый запуск для проверки кода.
QUICK_MODE = False

# Пересоздавать файлы даже при повторном запуске.
FORCE_REBUILD = False

SEED = 42
USE_HEURISTIC_GRAMMAR = True

if QUICK_MODE:
    SPLIT_SIZES = {
        "train": 500,
        "validation": 100,
        "test_synthetic": 100,
        "test_clean": 50,
    }
    RUN_NAME = "quick"
else:
    SPLIT_SIZES = {
        "train": 25_000,
        "validation": 1_000,
        "test_synthetic": 2_000,
        "test_clean": 500,
    }
    RUN_NAME = "full"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = Path("/content/drive/MyDrive/kazakh-gec")
else:
    PROJECT_DIR = Path("/content/kazakh-gec")

CACHE_DIR = PROJECT_DIR / "cache"
DATA_DIR = PROJECT_DIR / "data" / RUN_NAME

CACHE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_PARQUET = CACHE_DIR / "kazparc_raw.parquet"

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"

print("Project directory:", PROJECT_DIR)
print("Prepared data:", DATA_DIR)
print("Split sizes:", SPLIT_SIZES)

Mounted at /content/drive
Project directory: /content/drive/MyDrive/kazakh-gec
Prepared data: /content/drive/MyDrive/kazakh-gec/data/full
Split sizes: {'train': 25000, 'validation': 1000, 'test_synthetic': 2000, 'test_clean': 500}


In [ ]:
from datasets import load_dataset
from huggingface_hub import get_token, notebook_login

if RAW_PARQUET.exists() and not FORCE_REBUILD:
    print("Loading cached raw dataset:", RAW_PARQUET)

    raw = load_dataset(
        "parquet",
        data_files=str(RAW_PARQUET),
        split="train",
    )

    raw_source = str(RAW_PARQUET)

else:
    # Сначала пробуем получить HF_TOKEN из Colab Secrets.
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

    # Если секрет не найден — Colab покажет окно авторизации.
    if not hf_token and get_token() is None:
        notebook_login()
        hf_token = True

    print("Downloading issai/kazparc...")

    raw = load_dataset(
        "issai/kazparc",
        "kazparc_raw",
        split="train",
        token=hf_token if hf_token else True,
    )

    print("Saving raw dataset to:", RAW_PARQUET)
    raw.to_parquet(str(RAW_PARQUET))
    raw_source = "issai/kazparc / kazparc_raw"

required_columns = {"id", "kk", "domain"}
missing_columns = required_columns - set(raw.column_names)

assert not missing_columns, (
    f"Expected columns are missing: {missing_columns}. "
    f"Available columns: {raw.column_names}"
)

print(raw)
print(raw[0])

README.md:   0%|          | 0.00/19.3k [00:00<?, ?B/s]

kazparc/01_kazparc_all_entries.csv: reconstructing file:   0%|          |  0.00B /  226MB            

kazparc/01_kazparc_all_entries.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/371902 [00:00<?, ? examples/s]

Saving raw dataset to: /content/drive/MyDrive/kazakh-gec/cache/kazparc_raw.parquet


Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Dataset({
    features: ['id', 'kk', 'en', 'ru', 'tr', 'domain'],
    num_rows: 371902
})
{'id': 'LD024281LD', 'kk': 'Қауіпті қалдықтар трансшекаралық тасымалдау кезінде қаптау, таңбалау және тасу саласында жалпыға бірдей қабылданған халықаралық қағидалар мен нормаларға сәйкес қапталуға, таңбалануға және тасылуға тиіс.', 'en': 'During transboundary movement, hazardous waste shall be packaged, labelled and transported in accordance with generally accepted international rules and regulations in the field of packaging, labelling and transportation.', 'ru': 'При трансграничной перевозке опасные отходы должны упаковываться, маркироваться и транспортироваться в соответствии с общепринятыми международными правилами и нормами в области упаковки, маркировки и транспортировки.', 'tr': 'Tehlikeli atıkların sınır ötesi taşınması sırasında ambalajlama, etiketleme ve taşıma alanında genel kabul görmüş uluslararası kural ve düzenlemelere uygun olarak paketlenecek, etiketlenecek ve taşınacaktır.', 'do

In [ ]:
import hashlib
import re
import unicodedata

import pandas as pd

CYR = "А-Яа-яЁёӘәҒғҚқҢңӨөҰұҮүҺһІі"
CYRILLIC_CHAR_RE = re.compile(rf"[{CYR}]")
WHITESPACE_RE = re.compile(r"\s+")


def normalize_text(value):
    """Только Unicode NFC и нормализация пробелов."""
    if value is None or pd.isna(value):
        return ""

    value = unicodedata.normalize("NFC", str(value))
    return WHITESPACE_RE.sub(" ", value).strip()


def cyrillic_ratio(text):
    letters = [character for character in text if character.isalpha()]

    if not letters:
        return 0.0

    cyrillic_count = sum(
        CYRILLIC_CHAR_RE.fullmatch(character) is not None
        for character in letters
    )

    return cyrillic_count / len(letters)


def fallback_id(text):
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]
    return f"text-{digest}"


# Берём только нужные колонки. Переводы en/ru/tr не используются.
df = raw.select_columns(["id", "kk", "domain"]).to_pandas()

n_raw = len(df)

df["clean_text"] = df["kk"].map(normalize_text)
df["domain"] = df["domain"].map(normalize_text).replace("", "Unknown")
df["source_id"] = df["id"].map(normalize_text)

missing_id = df["source_id"].eq("")
df.loc[missing_id, "source_id"] = (
    df.loc[missing_id, "clean_text"].map(fallback_id)
)

df["word_count"] = df["clean_text"].str.split().str.len()
df["cyrillic_ratio"] = df["clean_text"].map(cyrillic_ratio)

# Пустые строки автоматически не проходят условие word_count >= 5.
df = df.loc[
    df["word_count"].between(5, 50)
    & (df["cyrillic_ratio"] >= 0.70)
].copy()

n_after_filter = len(df)

# Ключ нужен только для дедупликации.
# Сам clean_text сохраняет исходный регистр.
df["dedupe_key"] = df["clean_text"].map(str.casefold)

# Стабильная сортировка делает выбор дубликата воспроизводимым.
df = (
    df.sort_values(["dedupe_key", "source_id"], kind="stable")
      .drop_duplicates(subset="dedupe_key", keep="first")
      .reset_index(drop=True)
)

n_after_deduplication = len(df)
required_total = sum(SPLIT_SIZES.values())

assert len(df) >= required_total, (
    f"Need {required_total:,} unique sentences, "
    f"but only {len(df):,} passed filtering."
)

assert df["clean_text"].ne("").all()
assert not df["dedupe_key"].duplicated().any()

print({
    "raw_rows": n_raw,
    "after_filtering": n_after_filter,
    "after_deduplication": n_after_deduplication,
})

print("\nDomains:")
print(df["domain"].value_counts())

{'raw_rows': 371902, 'after_filtering': 312432, 'after_deduplication': 307622}

Domains:
domain
mass_media     109158
general         68069
legal_docs      66811
edu_and_sci     33495
fiction         30089
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split


def stratified_take(frame, number, random_state):
    if len(frame) < number:
        raise ValueError(
            f"Requested {number:,} rows, but only {len(frame):,} remain."
        )

    if len(frame) == number:
        return frame.reset_index(drop=True), frame.iloc[0:0].copy()

    selected, remainder = train_test_split(
        frame,
        train_size=number,
        random_state=random_state,
        shuffle=True,
        stratify=frame["domain"],
    )

    return (
        selected.reset_index(drop=True),
        remainder.reset_index(drop=True),
    )


total_required = sum(SPLIT_SIZES.values())

# Сначала выбираем общий фиксированный pool.
pool, unused = stratified_take(df, total_required, SEED)

train_base, remainder = stratified_take(
    pool,
    SPLIT_SIZES["train"],
    SEED + 1,
)

validation_base, remainder = stratified_take(
    remainder,
    SPLIT_SIZES["validation"],
    SEED + 2,
)

test_synthetic_base, test_clean_base = stratified_take(
    remainder,
    SPLIT_SIZES["test_synthetic"],
    SEED + 3,
)

base_splits = {
    "train": train_base,
    "validation": validation_base,
    "test_synthetic": test_synthetic_base,
    "test_clean": test_clean_base,
}

actual_sizes = {
    name: len(frame)
    for name, frame in base_splits.items()
}

assert actual_sizes == SPLIT_SIZES

# Проверка, что одинаковый нормализованный текст
# не появился в разных частях.
all_clean = pd.concat(
    [
        frame.assign(split=name)
        for name, frame in base_splits.items()
    ],
    ignore_index=True,
)

assert not all_clean["dedupe_key"].duplicated().any(), (
    "Clean-text leakage between splits!"
)

print("Split sizes:", actual_sizes)

for split_name, frame in base_splits.items():
    print(f"\n{split_name}:")
    print(
        (frame["domain"].value_counts(normalize=True) * 100)
        .round(1)
    )

Split sizes: {'train': 25000, 'validation': 1000, 'test_synthetic': 2000, 'test_clean': 500}

train:
domain
mass_media     35.5
general        22.1
legal_docs     21.7
edu_and_sci    10.9
fiction         9.8
Name: proportion, dtype: float64

validation:
domain
mass_media     35.5
general        22.1
legal_docs     21.7
edu_and_sci    10.9
fiction         9.8
Name: proportion, dtype: float64

test_synthetic:
domain
mass_media     35.5
general        22.2
legal_docs     21.7
edu_and_sci    10.9
fiction         9.8
Name: proportion, dtype: float64

test_clean:
domain
mass_media     35.4
general        22.2
legal_docs     21.8
edu_and_sci    10.8
fiction         9.8
Name: proportion, dtype: float64


In [ ]:
import random
from collections import Counter

WORD_RE = re.compile(rf"[{CYR}]+")


def stable_seed(*parts):
    payload = "\x1f".join(map(str, parts)).encode("utf-8")
    digest = hashlib.sha256(payload).digest()

    return int.from_bytes(digest[:8], "big")


def replace_span(text, start, end, replacement):
    return text[:start] + replacement + text[end:]


KAZAKH_LETTER_LOSS = {
    "ә": "а",
    "ғ": "г",
    "қ": "к",
    "ң": "н",
    "ө": "о",
    "ұ": "у",
    "ү": "у",
    "һ": "х",
    "і": "и",
}


def kazakh_letter_error(text, rng):
    positions = [
        index
        for index, character in enumerate(text)
        if character.lower() in KAZAKH_LETTER_LOSS
    ]

    if not positions:
        return None

    index = rng.choice(positions)
    old = text[index]
    new = KAZAKH_LETTER_LOSS[old.lower()]

    if old.isupper():
        new = new.upper()

    return (
        replace_span(text, index, index + 1, new),
        f"{old}->{new}",
    )


def typo_error(text, rng):
    words = [
        match
        for match in WORD_RE.finditer(text)
        if len(match.group()) >= 5
    ]

    if not words:
        return None

    match = rng.choice(words)
    word = match.group()

    swaps = [
        index
        for index in range(1, len(word) - 2)
        if word[index] != word[index + 1]
    ]

    if swaps and rng.random() < 0.5:
        index = rng.choice(swaps)

        changed_word = (
            word[:index]
            + word[index + 1]
            + word[index]
            + word[index + 2:]
        )

        detail = "adjacent_character_swap"

    else:
        index = rng.randrange(1, len(word) - 1)

        changed_word = (
            word[:index]
            + word[index + 1:]
        )

        detail = "character_deletion"

    return (
        replace_span(
            text,
            match.start(),
            match.end(),
            changed_word,
        ),
        detail,
    )


def missing_space_error(text, rng):
    matches = list(
        re.finditer(
            rf"(?<=[{CYR}]) (?=[{CYR}])",
            text,
        )
    )

    if not matches:
        return None

    match = rng.choice(matches)

    return (
        replace_span(text, match.start(), match.end(), ""),
        "deleted_space",
    )


def double_space_error(text, rng):
    matches = list(
        re.finditer(r"(?<=\S) (?=\S)", text)
    )

    if not matches:
        return None

    match = rng.choice(matches)

    return (
        replace_span(text, match.start(), match.end(), "  "),
        "duplicated_space",
    )


def final_punctuation_error(text, rng):
    match = re.search(
        r"[.!?…](?=[»”\"'’)\]]*\s*$)",
        text,
    )

    if not match:
        return None

    return (
        replace_span(text, match.start(), match.end(), ""),
        "deleted_final_punctuation",
    )


def initial_case_error(text, rng):
    match = WORD_RE.search(text)

    if not match:
        return None

    word = match.group()
    first_character = word[0]

    # Не повреждаем аббревиатуры вроде ҚР.
    if (
        not first_character.isupper()
        or (len(word) > 1 and word.isupper())
    ):
        return None

    return (
        replace_span(
            text,
            match.start(),
            match.start() + 1,
            first_character.lower(),
        ),
        "lowercased_first_letter",
    )


QUESTION_PARTICLE_RE = re.compile(
    rf" (?P<particle>ма|ме|ба|бе|па|пе)"
    rf"(?=\?(?:[»”\"'’)\]]*)?$)",
    re.IGNORECASE,
)


def question_particle_error(text, rng):
    match = QUESTION_PARTICLE_RE.search(text)

    if not match:
        return None

    return (
        replace_span(text, match.start(), match.start() + 1, ""),
        f"joined_{match.group('particle')}",
    )


def hyphen_error(text, rng):
    matches = list(
        re.finditer(
            rf"(?<=[0-9{CYR}])-(?=[{CYR}])",
            text,
        )
    )

    if not matches:
        return None

    match = rng.choice(matches)

    return (
        replace_span(text, match.start(), match.end(), " "),
        "hyphen_replaced_with_space",
    )

In [ ]:
FRONT_VOWELS = set("әеіөү")
BACK_VOWELS = set("аоұы")

SUFFIX_FLIPS = {
    "лар": ("лер", "back"),
    "лер": ("лар", "front"),
    "дар": ("дер", "back"),
    "дер": ("дар", "front"),
    "тар": ("тер", "back"),
    "тер": ("тар", "front"),

    "ның": ("нің", "back"),
    "нің": ("ның", "front"),
    "дың": ("дің", "back"),
    "дің": ("дың", "front"),
    "тың": ("тің", "back"),
    "тің": ("тың", "front"),

    "ны": ("ні", "back"),
    "ні": ("ны", "front"),
    "ды": ("ді", "back"),
    "ді": ("ды", "front"),
    "ты": ("ті", "back"),
    "ті": ("ты", "front"),

    "ға": ("ге", "back"),
    "ге": ("ға", "front"),
    "қа": ("ке", "back"),
    "ке": ("қа", "front"),

    "да": ("де", "back"),
    "де": ("да", "front"),
    "та": ("те", "back"),
    "те": ("та", "front"),

    "дан": ("ден", "back"),
    "ден": ("дан", "front"),
    "тан": ("тен", "back"),
    "тен": ("тан", "front"),
}

SORTED_SUFFIXES = sorted(
    SUFFIX_FLIPS,
    key=len,
    reverse=True,
)


def suffix_harmony_error(text, rng):
    candidates = []

    for match in WORD_RE.finditer(text):
        word = match.group()
        lower_word = word.lower()

        for suffix in SORTED_SUFFIXES:
            if not lower_word.endswith(suffix):
                continue

            stem = lower_word[:-len(suffix)]

            if len(stem) < 3:
                break

            last_vowel = next(
                (
                    character
                    for character in reversed(stem)
                    if character in FRONT_VOWELS | BACK_VOWELS
                ),
                None,
            )

            if last_vowel is None:
                break

            wrong_suffix, original_harmony = SUFFIX_FLIPS[suffix]

            stem_harmony = (
                "front"
                if last_vowel in FRONT_VOWELS
                else "back"
            )

            # Меняем только если исходный вариант согласуется
            # с гармонией основы.
            if original_harmony == stem_harmony:
                start = match.end() - len(suffix)
                original_suffix = text[start:match.end()]

                if original_suffix.isupper():
                    wrong_suffix = wrong_suffix.upper()

                candidates.append(
                    (
                        start,
                        match.end(),
                        wrong_suffix,
                        f"{original_suffix}->{wrong_suffix}",
                    )
                )

            break

    if not candidates:
        return None

    start, end, replacement, detail = rng.choice(candidates)

    return (
        replace_span(text, start, end, replacement),
        detail,
    )


PERSON_ENDING_FLIPS = {
    "мен": {
        "дым": "дың",
        "дім": "дің",
        "тым": "тың",
        "тім": "тің",
        "мын": "сың",
        "мін": "сің",
        "бын": "сың",
        "бін": "сің",
        "пын": "сың",
        "пін": "сің",
    },
    "сен": {
        "дың": "дым",
        "дің": "дім",
        "тың": "тым",
        "тің": "тім",
        "сың": "мын",
        "сің": "мін",
    },
    "біз": {
        "мыз": "сыз",
        "міз": "сіз",
        "быз": "сыз",
        "біз": "сіз",
        "пыз": "сыз",
        "піз": "сіз",
    },
    "сендер": {
        "сыңдар": "мыз",
        "сіңдер": "міз",
    },
}


def person_agreement_error(text, rng):
    words = list(WORD_RE.finditer(text))

    if len(words) < 2:
        return None

    pronoun = words[0].group().lower()

    if pronoun not in PERSON_ENDING_FLIPS:
        return None

    predicate = words[-1]

    # После предполагаемого сказуемого могут идти только знаки.
    if re.search(
        rf"[{CYR}A-Za-z0-9]",
        text[predicate.end():],
    ):
        return None

    predicate_lower = predicate.group().lower()
    endings = PERSON_ENDING_FLIPS[pronoun]

    for old_ending in sorted(endings, key=len, reverse=True):
        if (
            predicate_lower.endswith(old_ending)
            and len(predicate_lower) - len(old_ending) >= 2
        ):
            new_ending = endings[old_ending]
            start = predicate.end() - len(old_ending)
            original_ending = text[start:predicate.end()]

            if original_ending.isupper():
                new_ending = new_ending.upper()

            return (
                replace_span(
                    text,
                    start,
                    predicate.end(),
                    new_ending,
                ),
                f"{pronoun}:{old_ending}->{new_ending}",
            )

    return None

In [ ]:
ERROR_RULES = [
    # Специальные правила стоят раньше общих.
    ("question_particle_join", question_particle_error, False),
    ("person_agreement_heuristic", person_agreement_error, True),
    ("suffix_harmony_heuristic", suffix_harmony_error, True),

    ("kazakh_letter_loss", kazakh_letter_error, False),
    ("typo", typo_error, False),
    ("hyphen_to_space", hyphen_error, False),
    ("missing_final_punctuation", final_punctuation_error, False),
    ("initial_case", initial_case_error, False),
    ("missing_space", missing_space_error, False),
    ("double_space", double_space_error, False),
]


def get_error_candidates(text, split_name, source_id):
    candidates = []
    seen_outputs = set()

    for error_name, error_function, heuristic in ERROR_RULES:
        if heuristic and not USE_HEURISTIC_GRAMMAR:
            continue

        rule_rng = random.Random(
            stable_seed(
                SEED,
                split_name,
                source_id,
                error_name,
            )
        )

        result = error_function(text, rule_rng)

        if result is None:
            continue

        corrupted_text, detail = result

        if (
            corrupted_text == text
            or corrupted_text in seen_outputs
        ):
            continue

        seen_outputs.add(corrupted_text)

        candidates.append(
            (
                error_name,
                corrupted_text,
                detail,
                heuristic,
            )
        )

    return candidates


def build_pairs(base, split_name, identity_fraction=0.0):
    random_state = (
        stable_seed(SEED, split_name, "row_order")
        % (2**32 - 1)
    )

    work = (
        base.sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    identity_count = round(
        len(work) * identity_fraction
    )

    error_counts = Counter()
    records = []

    for row_index, row in enumerate(work.itertuples(index=False)):
        row_seed = stable_seed(
            SEED,
            split_name,
            row.source_id,
        )

        if row_index < identity_count:
            error_name = "clean_identity"
            corrupted_text = row.clean_text
            detail = ""
            heuristic = False

        else:
            options = get_error_candidates(
                row.clean_text,
                split_name,
                row.source_id,
            )

            if not options:
                raise RuntimeError(
                    "No error could be generated for: "
                    f"{row.clean_text!r}"
                )

            # Выбираем наименее представленный применимый тип.
            # Это делает распределение ошибок более равномерным.
            smallest_count = min(
                error_counts[item[0]]
                for item in options
            )

            tied_options = [
                item
                for item in options
                if error_counts[item[0]] == smallest_count
            ]

            choice_rng = random.Random(row_seed)

            (
                error_name,
                corrupted_text,
                detail,
                heuristic,
            ) = choice_rng.choice(tied_options)

            error_counts[error_name] += 1

        records.append({
            "source_id": str(row.source_id),
            "domain": str(row.domain),
            "split": split_name,
            "clean_text": row.clean_text,
            "corrupted_text": corrupted_text,
            "error_type": error_name,
            "error_detail": detail,
            "number_of_edits": (
                0 if error_name == "clean_identity" else 1
            ),
            "seed": row_seed,
            "needs_manual_review": heuristic,
        })

    return pd.DataFrame.from_records(records)


# Train: 20% уже правильных предложений, 80% с одной ошибкой.
train_pairs = build_pairs(
    train_base,
    "train",
    identity_fraction=0.20,
)

# Validation/test: ровно одна ошибка на предложение.
validation_pairs = build_pairs(
    validation_base,
    "validation",
    identity_fraction=0.0,
)

test_synthetic_pairs = build_pairs(
    test_synthetic_base,
    "test_synthetic",
    identity_fraction=0.0,
)

# Контроль переисправлений: только правильные предложения.
test_clean_pairs = build_pairs(
    test_clean_base,
    "test_clean",
    identity_fraction=1.0,
)

prepared_frames = {
    "train": train_pairs,
    "validation": validation_pairs,
    "test_synthetic": test_synthetic_pairs,
    "test_clean": test_clean_pairs,
}

for split_name, frame in prepared_frames.items():
    print(f"\n{split_name}: {len(frame):,}")
    print(frame["error_type"].value_counts())


train: 25,000
error_type
clean_identity                5000
missing_final_punctuation     2433
typo                          2433
kazakh_letter_loss            2433
missing_space                 2433
initial_case                  2433
double_space                  2433
suffix_harmony_heuristic      2432
hyphen_to_space               2431
person_agreement_heuristic     307
question_particle_join         232
Name: count, dtype: int64

validation: 1,000
error_type
missing_space                 122
kazakh_letter_loss            122
double_space                  122
typo                          122
missing_final_punctuation     121
suffix_harmony_heuristic      121
initial_case                  121
hyphen_to_space               119
person_agreement_heuristic     20
question_particle_join         10
Name: count, dtype: int64

test_synthetic: 2,000
error_type
double_space                  244
typo                          244
suffix_harmony_heuristic      244
initial_case                  2

In [ ]:
import json
from datetime import datetime, timezone

import datasets as datasets_package
import pyarrow
import sklearn
from datasets import Dataset

actual_sizes = {
    name: len(frame)
    for name, frame in prepared_frames.items()
}

assert actual_sizes == SPLIT_SIZES

combined = pd.concat(
    prepared_frames.values(),
    ignore_index=True,
)

# Предложения не пересекаются между частями.
combined_keys = combined["clean_text"].map(
    lambda text: normalize_text(text).casefold()
)

assert not combined_keys.duplicated().any(), (
    "Data leakage detected after corruption!"
)

# Identity examples должны совпадать.
identity_mask = train_pairs["number_of_edits"].eq(0)

assert (
    train_pairs.loc[identity_mask, "clean_text"]
    == train_pairs.loc[identity_mask, "corrupted_text"]
).all()

# Остальные train-примеры должны отличаться.
assert (
    train_pairs.loc[~identity_mask, "clean_text"]
    != train_pairs.loc[~identity_mask, "corrupted_text"]
).all()

# Validation и synthetic test содержат ровно одну ошибку.
assert validation_pairs["number_of_edits"].eq(1).all()
assert test_synthetic_pairs["number_of_edits"].eq(1).all()

assert (
    validation_pairs["clean_text"]
    != validation_pairs["corrupted_text"]
).all()

assert (
    test_synthetic_pairs["clean_text"]
    != test_synthetic_pairs["corrupted_text"]
).all()

# Clean control полностью состоит из identity-пар.
assert test_clean_pairs["number_of_edits"].eq(0).all()

assert (
    test_clean_pairs["clean_text"]
    == test_clean_pairs["corrupted_text"]
).all()

output_paths = {}

for split_name, frame in prepared_frames.items():
    output_path = DATA_DIR / f"{split_name}.parquet"

    dataset = Dataset.from_pandas(
        frame,
        preserve_index=False,
    )

    dataset.to_parquet(str(output_path))
    output_paths[split_name] = output_path

    size_mb = output_path.stat().st_size / (1024 ** 2)

    print(
        f"Saved {split_name}: "
        f"{output_path} ({size_mb:.2f} MiB)"
    )


def file_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source": raw_source,
    "source_dataset": "issai/kazparc",
    "source_config": "kazparc_raw",
    "raw_fingerprint": getattr(raw, "_fingerprint", None),
    "global_seed": SEED,
    "quick_mode": QUICK_MODE,
    "use_heuristic_grammar": USE_HEURISTIC_GRAMMAR,
    "normalization": (
        "Unicode NFC; collapse whitespace; strip"
    ),
    "filter": {
        "minimum_words": 5,
        "maximum_words": 50,
        "minimum_cyrillic_ratio": 0.70,
    },
    "deduplication": (
        "normalized clean_text with casefold"
    ),
    "counts": SPLIT_SIZES,
    "preprocessing_counts": {
        "raw": n_raw,
        "after_filter": n_after_filter,
        "after_deduplication": n_after_deduplication,
    },
    "training_mix": {
        "clean_identity": 0.20,
        "one_synthetic_error": 0.80,
    },
    "versions": {
        "datasets": datasets_package.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "files": {
        name: {
            "path": str(path),
            "sha256": file_sha256(path),
        }
        for name, path in output_paths.items()
    },
}

manifest_path = DATA_DIR / "data_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved manifest:", manifest_path)
print("All validation checks passed.")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved train: /content/drive/MyDrive/kazakh-gec/data/full/train.parquet (5.16 MiB)


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved validation: /content/drive/MyDrive/kazakh-gec/data/full/validation.parquet (0.21 MiB)


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved test_synthetic: /content/drive/MyDrive/kazakh-gec/data/full/test_synthetic.parquet (0.42 MiB)


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved test_clean: /content/drive/MyDrive/kazakh-gec/data/full/test_clean.parquet (0.11 MiB)
Saved manifest: /content/drive/MyDrive/kazakh-gec/data/full/data_manifest.json
All validation checks passed.


In [ ]:
from datasets import load_dataset

prepared = load_dataset(
    "parquet",
    data_files={
        "train": str(DATA_DIR / "train.parquet"),
        "validation": str(DATA_DIR / "validation.parquet"),
        "test_synthetic": str(
            DATA_DIR / "test_synthetic.parquet"
        ),
        "test_clean": str(
            DATA_DIR / "test_clean.parquet"
        ),
    },
)

# Эти переменные уже существуют и готовы для обучения.
train_dataset = prepared["train"]
validation_dataset = prepared["validation"]
test_dataset = prepared["test_synthetic"]
clean_control_dataset = prepared["test_clean"]

print(prepared)
print("\nFirst train example:")
print(train_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test_synthetic split: 0 examples [00:00, ? examples/s]

Generating test_clean split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_id', 'domain', 'split', 'clean_text', 'corrupted_text', 'error_type', 'error_detail', 'number_of_edits', 'seed', 'needs_manual_review'],
        num_rows: 25000
    })
    validation: Dataset({
        features: ['source_id', 'domain', 'split', 'clean_text', 'corrupted_text', 'error_type', 'error_detail', 'number_of_edits', 'seed', 'needs_manual_review'],
        num_rows: 1000
    })
    test_synthetic: Dataset({
        features: ['source_id', 'domain', 'split', 'clean_text', 'corrupted_text', 'error_type', 'error_detail', 'number_of_edits', 'seed', 'needs_manual_review'],
        num_rows: 2000
    })
    test_clean: Dataset({
        features: ['source_id', 'domain', 'split', 'clean_text', 'corrupted_text', 'error_type', 'error_detail', 'number_of_edits', 'seed', 'needs_manual_review'],
        num_rows: 500
    })
})

First train example:
{'source_id': 'GN000616GN', 'domain': 'general', 'split': 'train', 'clean_text': '

In [ ]:
from IPython.display import display

preview = (
    test_synthetic_pairs
    .sort_values(
        ["error_type", "source_id"],
        kind="stable",
    )
    .groupby("error_type", group_keys=False)
    .head(3)
)

display(
    preview[[
        "error_type",
        "error_detail",
        "corrupted_text",
        "clean_text",
        "domain",
        "needs_manual_review",
    ]]
)

,error_type,error_detail,corrupted_text,clean_text,domain,needs_manual_review
913,double_space,duplicated_space,Сіз ұялу мен сенімсіздіктен арылыңыз.,Сіз ұялу мен сенімсіздіктен арылыңыз.,edu_and_sci,False
715,double_space,duplicated_space,Сол кезде қарындасымнан екі жас үлкен болғанм...,Сол кезде қарындасымнан екі жас үлкен болғанмы...,edu_and_sci,False
1323,double_space,duplicated_space,Ерекше жұп үшін үйлену тойы туралы хабарлама ж...,Ерекше жұп үшін үйлену тойы туралы хабарлама ж...,edu_and_sci,False
685,hyphen_to_space,hyphen_replaced_with_space,Масалар кез келген ауруды таси алады: безгекте...,Масалар кез-келген ауруды таси алады: безгекте...,edu_and_sci,False
343,hyphen_to_space,hyphen_replaced_with_space,"Шындығында, біз Лос Анджелеске «Америкаға көше...","Шындығында, біз Лос-Анджелеске «Америкаға көше...",edu_and_sci,False
274,hyphen_to_space,hyphen_replaced_with_space,Бұл менің 31 ші туған күнім еді – бұл идеяны а...,Бұл менің 31-ші туған күнім еді – бұл идеяны а...,edu_and_sci,False
1644,initial_case,lowercased_first_letter,бірақ одан да көп миыңыз өлшенетіндей үлкейеді.,Бірақ одан да көп миыңыз өлшенетіндей үлкейеді.,edu_and_sci,False
1968,initial_case,lowercased_first_letter,ол нәрестені жақын маңдағы ауылдық емханаға ап...,Ол нәрестені жақын маңдағы ауылдық емханаға ап...,edu_and_sci,False
1114,initial_case,lowercased_first_letter,бірақ білім берудің мәні адамдардың оқығанында.,Бірақ білім берудің мәні адамдардың оқығанында.,edu_and_sci,False
649,kazakh_letter_loss,і->и,"Мысалы, ешкім бір уақытта еки нотаны орындай а...","Мысалы, ешкім бір уақытта екі нотаны орындай а...",edu_and_sci,False


In [ ]:
import subprocess
import sys


subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==5.14.1",
        "trl==1.9.0",
        "peft==0.19.1",
        "bitsandbytes==0.49.2",
        "accelerate==1.14.0",
        "datasets==5.0.0",
        "rapidfuzz>=3.14,<4",
    ]
)
print("Installed. Now choose Runtime > Restart session, then continue below.")

Installed. Now choose Runtime > Restart session, then continue below.


In [ ]:
from pathlib import Path
import gc
import importlib.metadata as metadata
import json
import os
import random
import time
import unicodedata

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from google.colab import drive


assert torch.cuda.is_available(), (
    "No GPU is attached. Choose Runtime > Change runtime type > T4 GPU."
)

drive.mount("/content/drive")

SEED = 42
MODEL_ID = "Qwen/Qwen3-0.6B"
MAX_LENGTH = 384

SYSTEM_PROMPT = (
    "Сен қазақ тіліндегі мәтінді түзететін редакторсың. "
    "Мағынасын өзгертпе. "
    "Жауапта тек түзетілген мәтінді бер, түсіндірме қоспа."
)
USER_PREFIX = "Қазақша мәтінді түзет:\n"

# Change only this path if your preparation notebook saved the files elsewhere.
PROJECT_DIR = Path("/content/drive/MyDrive/kazakh-gec")
DATA_DIR = PROJECT_DIR / "data" / "full"
MODEL_DIR = PROJECT_DIR / "models" / "qwen3-0.6b-kazakh-gec"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
ADAPTER_DIR = MODEL_DIR / "final"
RESULTS_DIR = PROJECT_DIR / "evaluation"

for directory in (MODEL_DIR, CHECKPOINT_DIR, ADAPTER_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

required_files = {
    "train": DATA_DIR / "train.parquet",
    "validation": DATA_DIR / "validation.parquet",
    "test_synthetic": DATA_DIR / "test_synthetic.parquet",
    "test_clean": DATA_DIR / "test_clean.parquet",
}
missing_files = [str(path) for path in required_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "These prepared files were not found:\n"
        + "\n".join(missing_files)
        + "\nSet DATA_DIR to the folder used by your corruption notebook."
    )

dataset = load_dataset(
    "parquet",
    data_files={name: str(path) for name, path in required_files.items()},
)

required_columns = {"corrupted_text", "clean_text"}
for split_name in ("train", "validation", "test_synthetic"):
    missing_columns = required_columns - set(dataset[split_name].column_names)
    if missing_columns:
        raise ValueError(
            f"{split_name} is missing {sorted(missing_columns)}. "
            f"Found {dataset[split_name].column_names}."
        )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}; VRAM: {gpu.total_memory / 2**30:.1f} GiB")
print(dataset)
print("First pair:")
print("INPUT: ", dataset["train"][0]["corrupted_text"])
print("TARGET:", dataset["train"][0]["clean_text"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4; VRAM: 14.6 GiB
DatasetDict({
    train: Dataset({
        features: ['source_id', 'domain', 'split', 'clean_text', 'corrupted_text', 'error_type', 'error_detail', 'number_of_edits', 'seed', 'needs_manual_review'],
        num_rows: 25000
    })
    validation: Dataset({
        features: ['source_id', 'domain', 'split', 'clean_text', 'corrupted_text', 'error_type', 'error_detail', 'number_of_edits', 'seed', 'needs_manual_review'],
        num_rows: 1000
    })
    test_synthetic: Dataset({
        features: ['source_id', 'domain', 'split', 'clean_text', 'corrupted_text', 'error_type', 'error_detail', 'number_of_edits', 'seed', 'needs_manual_review'],
        num_rows: 2000
    })
    test_clean: Dataset({
        features: ['source_id', 'domain', 'split', 'clean_text', 'corrupted_text', 'error_type', 'error_detail', 'number_of_edits', 'seed', '

In [ ]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def make_correction_messages(text):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PREFIX + str(text)},
    ]


def make_prompt_text(text):
    return tokenizer.apply_chat_template(
        make_correction_messages(text),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def convert_to_sft(row):
    return {
        "prompt": make_prompt_text(row["corrupted_text"]),
        "completion": row["clean_text"],
    }


def prepare_sft_split(split, split_name):
    converted = split.map(
        convert_to_sft,
        remove_columns=split.column_names,
        desc=f"Formatting {split_name}",
    )

    def add_token_length(row):
        full_text = row["prompt"] + row["completion"] + tokenizer.eos_token
        return {
            "n_tokens": len(
                tokenizer(full_text, add_special_tokens=False)["input_ids"]
            )
        }

    converted = converted.map(
        add_token_length,
        desc=f"Measuring {split_name} lengths",
    )
    original_size = len(converted)
    converted = converted.filter(
        lambda row: row["n_tokens"] <= MAX_LENGTH,
        desc=f"Filtering long {split_name} rows",
    )
    print(
        f"{split_name}: kept {len(converted):,}; "
        f"removed {original_size - len(converted):,} rows over {MAX_LENGTH} tokens"
    )
    return converted.remove_columns("n_tokens")


sft_train = prepare_sft_split(dataset["train"], "train")
sft_validation = prepare_sft_split(dataset["validation"], "validation")

# These fixed rows are used for COPY, base-Qwen, and adapted-Qwen comparisons.
SYNTHETIC_EVAL_SIZE = 300
CLEAN_EVAL_SIZE = 100

synthetic_df = (
    dataset["test_synthetic"]
    .shuffle(seed=SEED)
    .select(range(min(SYNTHETIC_EVAL_SIZE, len(dataset["test_synthetic"]))))
    .to_pandas()
)
synthetic_df["evaluation_group"] = "synthetic_error"

clean_df = (
    dataset["test_clean"]
    .shuffle(seed=SEED)
    .select(range(min(CLEAN_EVAL_SIZE, len(dataset["test_clean"]))))
    .to_pandas()
)
if "corrupted_text" not in clean_df.columns:
    clean_df["corrupted_text"] = clean_df["clean_text"]
clean_df["evaluation_group"] = "clean_control"

evaluation_df = pd.concat([synthetic_df, clean_df], ignore_index=True)
evaluation_df.to_parquet(
    RESULTS_DIR / "fixed_evaluation_subset.parquet",
    index=False,
)

print("Fixed evaluation rows:", len(evaluation_df))
print("Rendered prompt example:\n")
print(sft_train[0]["prompt"] + sft_train[0]["completion"])

train: kept 22,895; removed 2,105 rows over 384 tokens
validation: kept 914; removed 86 rows over 384 tokens
Fixed evaluation rows: 400
Rendered prompt example:

<|im_start|>system
Сен қазақ тіліндегі мәтінді түзететін редакторсың. Мағынасын өзгертпе. Жауапта тек түзетілген мәтінді бер, түсіндірме қоспа.<|im_end|>
<|im_start|>user
Қазақша мәтінді түзет:
Кәмпит қосылған торт баланы туған күнімен құттықтап тұр.<|im_end|>
<|im_start|>assistant
<think>

</think>

Кәмпит қосылған торт баланы туған күнімен құттықтап тұр.


In [ ]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, BitsAndBytesConfig


major, _ = torch.cuda.get_device_capability(0)
use_bf16 = major >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)


def generate_corrections(model, texts, batch_size=8, max_new_tokens=192):
    previous_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    predictions = []
    model.eval()
    model.config.use_cache = True

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Generating corrections",
    ):
        batch_texts = texts[start : start + batch_size]
        prompts = [make_prompt_text(text) for text in batch_texts]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(model.device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        new_ids = output_ids[:, inputs["input_ids"].shape[1] :]
        predictions.extend(
            text.strip()
            for text in tokenizer.batch_decode(new_ids, skip_special_tokens=True)
        )

    tokenizer.padding_side = previous_padding_side
    return predictions


print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=compute_dtype,
    device_map={"": 0},
)

evaluation_df["copy_baseline"] = evaluation_df["corrupted_text"].astype(str)
evaluation_df["before_training"] = generate_corrections(
    base_model,
    evaluation_df["corrupted_text"].astype(str).tolist(),
)
evaluation_df.to_parquet(
    RESULTS_DIR / "predictions_before_training.parquet",
    index=False,
)
display(
    evaluation_df[
        ["error_type", "corrupted_text", "clean_text", "before_training"]
    ].head(12)
)

del base_model
gc.collect()
torch.cuda.empty_cache()

Loading base model...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Generating corrections:   0%|          | 0/50 [00:00<?, ?it/s]

,error_type,corrupted_text,clean_text,before_training
0,hyphen_to_space,"Бұл қаражат еңбекақыны көбейтуге, баспананы жа...","Бұл қаражат еңбекақыны көбейтуге, баспананы жа...",**Қазақша мәтінді түзет:** \nБұл қаражат еңбе...
1,missing_space,Артында келіншегі мен жалғыз ұлы жәненемерелер...,Артында келіншегі мен жалғыз ұлы және немереле...,**Артында келіншегі мен жалғыз ұлы жәненемерел...
2,typo,Мейрамхана асханасының ортаыснда орналасқан жа...,Мейрамхана асханасының ортасында орналасқан жа...,**Мейрамхана асханасының ортаыснда орналасқан ...
3,suffix_harmony_heuristic,Бірақ бейтаныс кісі мен жақын досы екеуі бірде...,Бірақ бейтаныс кісі мен жақын досы екеуі бірде...,**Қазақша мәтінді түзет:** \nБірақ бейтаныс к...
4,missing_final_punctuation,Тәуелсіз еліміздің өзінің жұлдызы болады,Тәуелсіз еліміздің өзінің жұлдызы болады.,"**Қазақша мәтінді түзет:** \n""Қазақша мәтінді..."
5,kazakh_letter_loss,Кейбір үй шаруашылықтары ушін бұл барлық отбас...,Кейбір үй шаруашылықтары үшін бұл барлық отбас...,**Қазақша мәтінді түзет:** \nКейбір үй шаруаш...
6,typo,Ормандағы теміржолен келе жатқан көк және сары...,Ормандағы теміржолмен келе жатқан көк және сар...,**Ормандағы теміржолен келе жатқан көк және са...
7,suffix_harmony_heuristic,Неліктан егер біреу оның қызының достары low-l...,Неліктен егер біреу оның қызының достары low-l...,**Қазақша мәтінді түзет:** \nҚазақ сөзінен ер...
8,kazakh_letter_loss,Еуропа сол жақ жоғарғы бурышта.,Еуропа сол жақ жоғарғы бұрышта.,"**Қазақша мәтінді түзет:** \n""Еуропа сол жақ ..."
9,hyphen_to_space,Себебі өмірде адам өзін өзі алға жетелеуі керек.,Себебі өмірде адам өзін-өзі алға жетелеуі керек.,**Қазақша мәтінді түзет:** \nҚазақ сәйкікінен...


In [ ]:
# QLoRA TRAINING CELL — run before the reload/evaluation cell

import time
import torch

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    EarlyStoppingCallback,
    TrainerCallback,
    set_seed,
)
from trl import SFTConfig, SFTTrainer


required_variables = [
    "MODEL_ID",
    "CHECKPOINT_DIR",
    "ADAPTER_DIR",
    "RESULTS_DIR",
    "MAX_LENGTH",
    "SEED",
    "tokenizer",
    "sft_train",
    "sft_validation",
    "quantization_config",
    "compute_dtype",
    "use_bf16",
]

missing = [name for name in required_variables if name not in globals()]
if missing:
    raise RuntimeError(
        f"Required variables are missing: {missing}. "
        "Rerun the setup, dataset, and prompt-formatting cells first."
    )

set_seed(SEED)

TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 8
MAX_TRAIN_SECONDS = 2 * 60 * 60 + 20 * 60


class TimeLimitCallback(TrainerCallback):
    def __init__(self, maximum_seconds):
        self.maximum_seconds = maximum_seconds
        self.started_at = None
        self.hit_limit = False

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.perf_counter() - self.started_at

        if elapsed >= self.maximum_seconds:
            self.hit_limit = True
            control.should_save = True
            control.should_training_stop = True

        return control


time_limit_callback = TimeLimitCallback(MAX_TRAIN_SECONDS)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

training_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=1.0,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    eos_token=tokenizer.eos_token,
    packing=False,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    logging_first_step=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

print("Loading Qwen for QLoRA training...")

from transformers import BitsAndBytesConfig

# Use one consistent compute type on all Colab NVIDIA GPUs.
compute_dtype = torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("Loading Qwen for QLoRA training...")

training_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map={"": 0},
)

training_model.config.use_cache = False

# Explicitly prepare the 4-bit model.
training_model = prepare_model_for_kbit_training(
    training_model,
    use_gradient_checkpointing=True,
)

# Add LoRA manually so we control its precision.
training_model = get_peft_model(
    training_model,
    lora_config,
)

# Optimizer parameters must remain FP32.
for parameter in training_model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.to(torch.float32)

trainable_dtypes = {
    str(parameter.dtype)
    for parameter in training_model.parameters()
    if parameter.requires_grad
}

print("Trainable parameter types:", trainable_dtypes)
assert trainable_dtypes == {"torch.float32"}, (
    f"Unexpected trainable parameter types: {trainable_dtypes}"
)

trainer = SFTTrainer(
    model=training_model,
    args=training_config,
    train_dataset=sft_train,
    eval_dataset=sft_validation,
    processing_class=tokenizer,
    # Do not pass peft_config here because LoRA is already attached.
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001,
        ),
        time_limit_callback,
    ],
)

trainer.model.print_trainable_parameters()

print("Starting fine-tuning...")
train_result = trainer.train()

print("Training stopped at step:", trainer.state.global_step)
print("Training epoch:", trainer.state.epoch)

# Save LoRA adapter
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(
    str(ADAPTER_DIR),
    safe_serialization=True,
)
tokenizer.save_pretrained(str(ADAPTER_DIR))

config_file = ADAPTER_DIR / "adapter_config.json"
adapter_file = ADAPTER_DIR / "adapter_model.safetensors"

print("adapter_config.json:", config_file.exists())
print("adapter_model.safetensors:", adapter_file.exists())
print("Saved files:", [path.name for path in ADAPTER_DIR.iterdir()])

assert config_file.exists(), "adapter_config.json was not saved."
assert adapter_file.exists(), "adapter_model.safetensors was not saved."

print("SUCCESS — adapter saved to:", ADAPTER_DIR)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading Qwen for QLoRA training...
Loading Qwen for QLoRA training...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Trainable parameter types: {'torch.float32'}


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
Starting fine-tuning...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.052077,0.060594,0.064895,707082.000000,0.987083
200,0.037135,0.051504,0.073085,1404269.000000,0.988971
300,0.043222,0.044699,0.056960,2105007.000000,0.990598
400,0.035357,0.042372,0.068304,2811811.000000,0.990995
500,0.037713,0.039774,0.058903,3514901.000000,0.991309
600,0.033126,0.038478,0.048551,4217001.000000,0.991639
700,0.038655,0.038257,0.052091,4923170.000000,0.991730
716,0.028299,0.038257,0.052091,5032592.000000,0.991730


Training stopped at step: 716
Training epoch: 1.0
adapter_config.json: True
adapter_model.safetensors: True
Saved files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
SUCCESS — adapter saved to: /content/drive/MyDrive/kazakh-gec/models/qwen3-0.6b-kazakh-gec/final


In [ ]:
# COMPLETE AFTER-TRAINING EVALUATION CELL

from pathlib import Path
import gc
import unicodedata

import pandas as pd
import torch
from google.colab import drive
from peft import PeftModel
from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


# --------------------------------------------------
# 1. Paths and model configuration
# --------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

MODEL_ID = "Qwen/Qwen3-0.6B"
MAX_LENGTH = 384

PROJECT_DIR = Path("/content/drive/MyDrive/kazakh-gec")
ADAPTER_DIR = (
    PROJECT_DIR
    / "models"
    / "qwen3-0.6b-kazakh-gec"
    / "final"
)
RESULTS_DIR = PROJECT_DIR / "evaluation"

BASELINE_PATH = RESULTS_DIR / "predictions_before_training.parquet"
COMBINED_PATH = RESULTS_DIR / "before_after_predictions.parquet"

SYSTEM_PROMPT = (
    "Сен қазақ тіліндегі мәтінді түзететін редакторсың. "
    "Мағынасын өзгертпе. "
    "Жауапта тек түзетілген мәтінді бер, түсіндірме қоспа."
)
USER_PREFIX = "Қазақша мәтінді түзет:\n"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert (ADAPTER_DIR / "adapter_config.json").exists(), (
    f"adapter_config.json was not found in {ADAPTER_DIR}"
)

assert BASELINE_PATH.exists(), (
    f"Baseline predictions were not found at {BASELINE_PATH}"
)

assert torch.cuda.is_available(), (
    "No GPU is connected. Select Runtime → Change runtime type → T4 GPU."
)

print("GPU:", torch.cuda.get_device_name(0))
print("Adapter:", ADAPTER_DIR)


# --------------------------------------------------
# 2. Restore tokenizer and saved baseline predictions
# --------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))
tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def make_prompt_text(text):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": USER_PREFIX + str(text),
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


evaluation_df = pd.read_parquet(BASELINE_PATH)

required_columns = {
    "corrupted_text",
    "clean_text",
    "before_training",
    "evaluation_group",
}

missing_columns = required_columns - set(evaluation_df.columns)

if missing_columns:
    raise ValueError(
        f"Baseline file is missing columns: {sorted(missing_columns)}"
    )

if "copy_baseline" not in evaluation_df.columns:
    evaluation_df["copy_baseline"] = evaluation_df["corrupted_text"].astype(str)

print("Evaluation examples:", len(evaluation_df))


# --------------------------------------------------
# 3. Release the training model from GPU
# --------------------------------------------------

for variable_name in (
    "trainer",
    "training_model",
    "trained_model",
    "base_model",
):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()


# --------------------------------------------------
# 4. Load the base model and trained LoRA adapter
# --------------------------------------------------

compute_dtype = torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("Loading base Qwen model...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map={"": 0},
)

print("Attaching the trained LoRA adapter...")

trained_model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
)
trained_model.eval()
trained_model.config.use_cache = True


# --------------------------------------------------
# 5. Generation function
# --------------------------------------------------

def generate_corrections(
    model,
    texts,
    batch_size=8,
    max_new_tokens=192,
):
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    predictions = []
    model_device = next(model.parameters()).device

    try:
        for start in tqdm(
            range(0, len(texts), batch_size),
            desc="Generating trained-model corrections",
        ):
            batch_texts = texts[start : start + batch_size]

            prompts = [
                make_prompt_text(text)
                for text in batch_texts
            ]

            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
            ).to(model_device)

            with torch.inference_mode():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    num_beams=1,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                )

            generated_only = generated[
                :,
                inputs["input_ids"].shape[1] :,
            ]

            decoded = tokenizer.batch_decode(
                generated_only,
                skip_special_tokens=True,
            )

            predictions.extend(
                output.strip()
                for output in decoded
            )

    finally:
        tokenizer.padding_side = old_padding_side

    return predictions


# --------------------------------------------------
# 6. Generate and save after-training predictions
# --------------------------------------------------

print("Generating after-training predictions...")
print("This will probably take around 15–20 minutes.")

evaluation_df["after_training"] = generate_corrections(
    trained_model,
    evaluation_df["corrupted_text"].astype(str).tolist(),
)

evaluation_df.to_parquet(
    COMBINED_PATH,
    index=False,
)

print("Predictions saved to:", COMBINED_PATH)


# --------------------------------------------------
# 7. Calculate before-versus-after metrics
# --------------------------------------------------

def normalize_for_evaluation(text):
    return unicodedata.normalize(
        "NFC",
        str(text),
    ).strip()


def calculate_metrics(frame, prediction_column):
    references = [
        normalize_for_evaluation(value)
        for value in frame["clean_text"]
    ]

    inputs = [
        normalize_for_evaluation(value)
        for value in frame["corrupted_text"]
    ]

    predictions = [
        normalize_for_evaluation(value)
        for value in frame[prediction_column]
    ]

    input_distances = [
        Levenshtein.distance(reference, source)
        for reference, source in zip(references, inputs)
    ]

    output_distances = [
        Levenshtein.distance(reference, prediction)
        for reference, prediction in zip(references, predictions)
    ]

    total_reference_characters = sum(
        max(len(reference), 1)
        for reference in references
    )

    number_of_examples = len(frame)

    return {
        "examples": number_of_examples,
        "exact_match_percent": round(
            100
            * sum(
                prediction == reference
                for prediction, reference in zip(
                    predictions,
                    references,
                )
            )
            / number_of_examples,
            2,
        ),
        "character_error_rate_percent": round(
            100
            * sum(output_distances)
            / total_reference_characters,
            2,
        ),
        "improved_percent": round(
            100
            * sum(
                output_distance < input_distance
                for output_distance, input_distance in zip(
                    output_distances,
                    input_distances,
                )
            )
            / number_of_examples,
            2,
        ),
        "harmed_percent": round(
            100
            * sum(
                output_distance > input_distance
                for output_distance, input_distance in zip(
                    output_distances,
                    input_distances,
                )
            )
            / number_of_examples,
            2,
        ),
        "copied_input_percent": round(
            100
            * sum(
                prediction == source
                for prediction, source in zip(
                    predictions,
                    inputs,
                )
            )
            / number_of_examples,
            2,
        ),
    }


stage_columns = {
    "copy": "copy_baseline",
    "before": "before_training",
    "after": "after_training",
}

metric_rows = []

for group_name, group in evaluation_df.groupby("evaluation_group"):
    for stage_name, prediction_column in stage_columns.items():
        metric_rows.append(
            {
                "group": group_name,
                "stage": stage_name,
                **calculate_metrics(
                    group,
                    prediction_column,
                ),
            }
        )

metrics_df = pd.DataFrame(metric_rows)

metrics_path = RESULTS_DIR / "evaluation_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)


# --------------------------------------------------
# 8. Calculate metrics for each corruption type
# --------------------------------------------------

if "error_type" in evaluation_df.columns:
    synthetic_rows = evaluation_df[
        evaluation_df["evaluation_group"] == "synthetic_error"
    ]

    per_error_rows = []

    for error_type, group in synthetic_rows.groupby("error_type"):
        for stage_name, prediction_column in stage_columns.items():
            per_error_rows.append(
                {
                    "error_type": error_type,
                    "stage": stage_name,
                    **calculate_metrics(
                        group,
                        prediction_column,
                    ),
                }
            )

    per_error_df = pd.DataFrame(per_error_rows)

    per_error_path = RESULTS_DIR / "per_error_metrics.csv"
    per_error_df.to_csv(per_error_path, index=False)
else:
    per_error_df = pd.DataFrame()
    per_error_path = None


# --------------------------------------------------
# 9. Display results
# --------------------------------------------------

print("\nOVERALL METRICS")
display(metrics_df)

if not per_error_df.empty:
    print("\nMETRICS BY ERROR TYPE")
    display(per_error_df)

example_columns = [
    column
    for column in [
        "error_type",
        "corrupted_text",
        "clean_text",
        "before_training",
        "after_training",
    ]
    if column in evaluation_df.columns
]

print("\nEXAMPLE CORRECTIONS")
display(evaluation_df[example_columns].head(20))


# --------------------------------------------------
# 10. Quick custom test
# --------------------------------------------------

CUSTOM_TEXT = "Мен бүгін университетке бардымба?"

custom_result = generate_corrections(
    trained_model,
    [CUSTOM_TEXT],
    batch_size=1,
    max_new_tokens=128,
)[0]

print("\nCUSTOM TEST")
print("Input: ", CUSTOM_TEXT)
print("Output:", custom_result)

print("\nEvaluation finished successfully.")
print("Overall metrics:", metrics_path)
print("Predictions:", COMBINED_PATH)

if per_error_path is not None:
    print("Per-error metrics:", per_error_path)

GPU: Tesla T4
Adapter: /content/drive/MyDrive/kazakh-gec/models/qwen3-0.6b-kazakh-gec/final
Evaluation examples: 400
Loading base Qwen model...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Attaching the trained LoRA adapter...
Generating after-training predictions...
This will probably take around 15–20 minutes.


Generating trained-model corrections:   0%|          | 0/50 [00:00<?, ?it/s]

...


EXAMPLE CORRECTIONS


,error_type,corrupted_text,clean_text,before_training,after_training
0,hyphen_to_space,"Бұл қаражат еңбекақыны көбейтуге, баспананы жа...","Бұл қаражат еңбекақыны көбейтуге, баспананы жа...",**Қазақша мәтінді түзет:** \nБұл қаражат еңбе...,"Бұл қаражат еңбекақыны көбейтуге, баспананы жа..."
1,missing_space,Артында келіншегі мен жалғыз ұлы жәненемерелер...,Артында келіншегі мен жалғыз ұлы және немереле...,**Артында келіншегі мен жалғыз ұлы жәненемерел...,Артында келіншегі мен жалғыз ұлы және менемере...
2,typo,Мейрамхана асханасының ортаыснда орналасқан жа...,Мейрамхана асханасының ортасында орналасқан жа...,**Мейрамхана асханасының ортаыснда орналасқан ...,Мейрамхана асханасының ортаыснда орналасқан жа...
3,suffix_harmony_heuristic,Бірақ бейтаныс кісі мен жақын досы екеуі бірде...,Бірақ бейтаныс кісі мен жақын досы екеуі бірде...,**Қазақша мәтінді түзет:** \nБірақ бейтаныс к...,Бірақ бейтаныс кісі мен жақын досы екеуі бірде...
4,missing_final_punctuation,Тәуелсіз еліміздің өзінің жұлдызы болады,Тәуелсіз еліміздің өзінің жұлдызы болады.,"**Қазақша мәтінді түзет:** \n""Қазақша мәтінді...",Тәуелсіз еліміздің өзінің жұлдызы болады.
5,kazakh_letter_loss,Кейбір үй шаруашылықтары ушін бұл барлық отбас...,Кейбір үй шаруашылықтары үшін бұл барлық отбас...,**Қазақша мәтінді түзет:** \nКейбір үй шаруаш...,Кейбір үй шаруашылықтары ушін бұл барлық отбас...
6,typo,Ормандағы теміржолен келе жатқан көк және сары...,Ормандағы теміржолмен келе жатқан көк және сар...,**Ормандағы теміржолен келе жатқан көк және са...,Ормандағы теміржолен келе жатқан көк және сары...
7,suffix_harmony_heuristic,Неліктан егер біреу оның қызының достары low-l...,Неліктен егер біреу оның қызының достары low-l...,**Қазақша мәтінді түзет:** \nҚазақ сөзінен ер...,Неліктан егер біреу оның қызының достары low-l...
8,kazakh_letter_loss,Еуропа сол жақ жоғарғы бурышта.,Еуропа сол жақ жоғарғы бұрышта.,"**Қазақша мәтінді түзет:** \n""Еуропа сол жақ ...",Еуропа сол жақ жоғарғы бұрышта.
9,hyphen_to_space,Себебі өмірде адам өзін өзі алға жетелеуі керек.,Себебі өмірде адам өзін-өзі алға жетелеуі керек.,**Қазақша мәтінді түзет:** \nҚазақ сәйкікінен...,Себебі өмірде адам өзін-өзі алға жетелеуі керек.


Generating trained-model corrections:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



CUSTOM TEST
Input:  Мен бүгін университетке бардымба?
Output: Мен бүгін университетке бардым ба?

Evaluation finished successfully.
Overall metrics: /content/drive/MyDrive/kazakh-gec/evaluation/evaluation_metrics.csv
Predictions: /content/drive/MyDrive/kazakh-gec/evaluation/before_after_predictions.parquet
Per-error metrics: /content/drive/MyDrive/kazakh-gec/evaluation/per_error_metrics.csv


STRICT AND CONTENT-ONLY METRICS


,group,evaluation_mode,stage,examples,exact_match_percent,character_error_rate_percent,improved_percent,harmed_percent,copied_input_percent
0,clean_control,strict_raw,copy,100,100.00,0.00,0.00,0.00,100.00
1,clean_control,strict_raw,before,100,0.00,129.46,0.00,100.00,0.00
2,clean_control,strict_raw,after,100,91.00,0.69,0.00,9.00,91.00
3,clean_control,content_only,copy,100,100.00,0.00,0.00,0.00,100.00
4,clean_control,content_only,before,100,10.00,107.78,0.00,90.00,10.00
5,clean_control,content_only,after,100,91.00,0.69,0.00,9.00,91.00
6,synthetic_error,strict_raw,copy,300,0.00,1.12,0.00,0.00,100.00
7,synthetic_error,strict_raw,before,300,0.00,127.65,0.00,100.00,0.00
8,synthetic_error,strict_raw,after,300,63.67,1.10,65.33,8.67,23.00
9,synthetic_error,content_only,copy,300,0.00,1.12,0.00,0.00,100.00



OUTPUT-FORMAT COMPLIANCE


,group,stage,format_compliance_percent,wrapper_removed_percent
0,clean_control,before,0.00,100.00
1,clean_control,after,100.00,0.00
2,synthetic_error,before,0.00,100.00
3,synthetic_error,after,99.67,0.33



CONTENT-ONLY METRICS BY ERROR TYPE


,error_type,stage,examples,exact_match_percent,character_error_rate_percent,improved_percent,harmed_percent,copied_input_percent
0,double_space,copy,34,0.00,1.04,0.00,0.00,100.00
1,double_space,before,34,5.88,113.72,5.88,94.12,0.00
2,double_space,after,34,94.12,0.83,94.12,2.94,0.00
3,hyphen_to_space,copy,38,0.00,0.66,0.00,0.00,100.00
4,hyphen_to_space,before,38,0.00,73.26,0.00,92.11,7.89
5,hyphen_to_space,after,38,50.00,2.31,50.00,5.26,44.74
6,initial_case,copy,34,0.00,1.06,0.00,0.00,100.00
7,initial_case,before,34,8.82,144.68,8.82,91.18,0.00
8,initial_case,after,34,94.12,0.06,94.12,0.00,5.88
9,kazakh_letter_loss,copy,38,0.00,1.16,0.00,0.00,100.00



MOST HARMFUL AFTER-TRAINING CHANGES


,evaluation_group,error_type,corrupted_text,clean_text,after_content,after_harm_delta
180,synthetic_error,hyphen_to_space,Кондоминиум объектісін басқаруға және кондомин...,Кондоминиум объектісін басқаруға және кондомин...,Кондоминиум объектісін басқаруға және кондомин...,111
393,clean_control,clean_identity,Ұсталған немесе қамаудағы тұлғаның ұсталу неме...,Ұсталған немесе қамаудағы тұлғаның ұсталу неме...,Ұсталған немесе қамаудағы тұлғаның ұсталу неме...,46
230,synthetic_error,suffix_harmony_heuristic,Қазақстан Республикасының газ және газбен жабд...,Қазақстан Республикасының газ және газбен жабд...,Қазақстан Республикасының газ және газбен жабд...,31
66,synthetic_error,double_space,Құбыржолдардың зақымдалу ықтималдығын болғызба...,Құбыржолдардың зақымдалу ықтималдығын болғызба...,Құбыржолдардың зақымдалу ықтималдығын болғызба...,25
337,clean_control,clean_identity,"Қатысушы мемлекеттер, төмендегі процессуалдық ...","Қатысушы мемлекеттер, төмендегі процессуалдық ...","Қатысушы мемлекеттер, төмендегі процессуалдық ...",11
121,synthetic_error,typo,"Өтінемін, несиені мүмкінідгінше тезірек жабыңыз.","Өтінемін, несиені мүмкіндігінше тезірек жабыңыз.","Өтінемін, несиені мүмкінің жағдынше тезірек жа...",6
335,clean_control,clean_identity,"Сот істi, шағымды, наразылықты қарауға әзірлеу...","Сот істi, шағымды, наразылықты қарауға әзірлеу...","Сот істi, шағымды, наразылықты қарауға әзірлеу...",5
160,synthetic_error,missing_space,Тамақтанубарысында суретке түсіп отырған отбасы.,Тамақтану барысында суретке түсіп отырған отбасы.,Тамақ тұбарысында суретке түсіп отырған отбасы.,4
255,synthetic_error,typo,Жауапты органдарға тіисті тапсырмаларын берді.,Жауапты органдарға тиісті тапсырмаларын берді.,ты органдарға тісті тапсырмаларын берді.,4
342,clean_control,clean_identity,"Жуынатын бөлмеде дәретхана, шұңғылша, ванна жә...","Жуынатын бөлмеде дәретхана, шұңғылша, ванна жә...","Жуынатын бөлмеде дәретхана, шұңғылша, ванна жә...",4



EXAMPLES WHERE THE BASE MODEL'S WRAPPER WAS REMOVED


,clean_text,before_training,before_content
0,"Бұл қаражат еңбекақыны көбейтуге, баспананы жа...",**Қазақша мәтінді түзет:** \nБұл қаражат еңбе...,"Бұл қаражат еңбекақыны көбейтуге, баспананы жа..."
1,Артында келіншегі мен жалғыз ұлы және немереле...,**Артында келіншегі мен жалғыз ұлы жәненемерел...,Артында келіншегі мен жалғыз ұлы жәненемерелер...
2,Мейрамхана асханасының ортасында орналасқан жа...,**Мейрамхана асханасының ортаыснда орналасқан ...,Мейрамхана асханасының ортаыснда орналасқан жа...
3,Бірақ бейтаныс кісі мен жақын досы екеуі бірде...,**Қазақша мәтінді түзет:** \nБірақ бейтаныс к...,Бірақ бейтаныс кісі мен жақын досы екеуі бірде...
4,Тәуелсіз еліміздің өзінің жұлдызы болады.,"**Қазақша мәтінді түзет:** \n""Қазақша мәтінді...","""Қазақша мәтінді түзет: Тәуелсіз еліміздің өзі..."
5,Кейбір үй шаруашылықтары үшін бұл барлық отбас...,**Қазақша мәтінді түзет:** \nКейбір үй шаруаш...,Кейбір үй шаруашылықтары ушін барлық отбасы бі...
6,Ормандағы теміржолмен келе жатқан көк және сар...,**Ормандағы теміржолен келе жатқан көк және са...,Ормандағы теміржолен келе жатқан көк және сары...
7,Неліктен егер біреу оның қызының достары low-l...,**Қазақша мәтінді түзет:** \nҚазақ сөзінен ер...,Қазақ сөзінен ерілген мәтінді түзетін. \n\n**...
8,Еуропа сол жақ жоғарғы бұрышта.,"**Қазақша мәтінді түзет:** \n""Еуропа сол жақ ...","""Еуропа сол жақ жоғарғы бурышта."" \n\n**Мағын..."
9,Себебі өмірде адам өзін-өзі алға жетелеуі керек.,**Қазақша мәтінді түзет:** \nҚазақ сәйкікінен...,Қазақ сәйкікінен өзін өзі алға жетелеуі керек....


In [ ]:
# STEP 1 — AUDIT THE V1 MODEL
# Safe: reads existing predictions and creates a new timestamped CSV.
# It does not load/train the model or overwrite existing artifacts.

%pip -q install rapidfuzz pyarrow

from datetime import datetime
from pathlib import Path
import unicodedata

import pandas as pd
from google.colab import drive
from rapidfuzz.distance import Levenshtein
from IPython.display import display


# --------------------------------------------------
# 1. Connect Drive and find predictions
# --------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/kazakh-gec")
EVALUATION_DIR = PROJECT_DIR / "evaluation"

candidate_files = [
    EVALUATION_DIR / "predictions_with_extracted_content.parquet",
    EVALUATION_DIR / "before_after_predictions.parquet",
]

prediction_path = next(
    (path for path in candidate_files if path.exists()),
    None,
)

if prediction_path is None:
    available = (
        sorted(path.name for path in EVALUATION_DIR.glob("*"))
        if EVALUATION_DIR.exists()
        else []
    )
    raise FileNotFoundError(
        "Prediction file was not found.\n"
        f"Checked: {[str(path) for path in candidate_files]}\n"
        f"Files currently available: {available}"
    )

df = pd.read_parquet(prediction_path).copy()
df.insert(0, "row_id", range(len(df)))

prediction_column = next(
    (
        column
        for column in ["after_content", "after_training"]
        if column in df.columns
    ),
    None,
)

if prediction_column is None:
    raise ValueError(
        "Neither 'after_content' nor 'after_training' exists. "
        f"Available columns: {sorted(df.columns)}"
    )

required_columns = {
    "clean_text",
    "corrupted_text",
    "evaluation_group",
}

missing = required_columns - set(df.columns)

if missing:
    raise ValueError(f"Required columns are missing: {sorted(missing)}")

if "error_type" not in df.columns:
    df["error_type"] = "unknown"

df["evaluation_group"] = (
    df["evaluation_group"].fillna("unknown").astype(str)
)
df["error_type"] = df["error_type"].fillna("unknown").astype(str)

print("Loaded:", prediction_path)
print("Examples:", len(df))
print("Prediction column:", prediction_column)


# --------------------------------------------------
# 2. Calculate audit measurements
# --------------------------------------------------

def normalize_text(value):
    if value is None or pd.isna(value):
        return ""

    # Do not collapse internal spaces:
    # double-space and missing-space errors must remain measurable.
    return unicodedata.normalize("NFC", str(value)).strip()


df["_reference"] = df["clean_text"].map(normalize_text)
df["_source"] = df["corrupted_text"].map(normalize_text)
df["_prediction"] = df[prediction_column].map(normalize_text)

df["input_distance"] = [
    Levenshtein.distance(reference, source)
    for reference, source in zip(df["_reference"], df["_source"])
]

df["output_distance"] = [
    Levenshtein.distance(reference, prediction)
    for reference, prediction in zip(
        df["_reference"],
        df["_prediction"],
    )
]

df["model_edit_distance"] = [
    Levenshtein.distance(source, prediction)
    for source, prediction in zip(
        df["_source"],
        df["_prediction"],
    )
]

df["harm_delta"] = df["output_distance"] - df["input_distance"]

df["exact"] = df["_prediction"] == df["_reference"]
df["copied_input"] = df["_prediction"] == df["_source"]
df["improved"] = df["harm_delta"] < 0
df["harmed"] = df["harm_delta"] > 0

df["length_ratio"] = [
    len(prediction) / max(len(source), 1)
    for source, prediction in zip(
        df["_source"],
        df["_prediction"],
    )
]

df["normalized_edit_ratio"] = [
    edit_distance / max(len(source), 1)
    for edit_distance, source in zip(
        df["model_edit_distance"],
        df["_source"],
    )
]

# These are provisional audit flags, not final guardrail thresholds.
df["catastrophic_candidate"] = (
    (df["harm_delta"] >= 10)
    | (df["normalized_edit_ratio"] > 0.30)
    | (df["length_ratio"] < 0.75)
    | (df["length_ratio"] > 1.25)
)


# --------------------------------------------------
# 3. Summaries
# --------------------------------------------------

def summarize(frame):
    reference_characters = sum(
        max(len(value), 1) for value in frame["_reference"]
    )

    positive_harm = frame.loc[frame["harmed"], "harm_delta"]

    return {
        "examples": len(frame),
        "exact_percent": round(100 * frame["exact"].mean(), 2),
        "cer_percent": round(
            100 * frame["output_distance"].sum()
            / max(reference_characters, 1),
            2,
        ),
        "improved_percent": round(
            100 * frame["improved"].mean(), 2
        ),
        "harmed_percent": round(
            100 * frame["harmed"].mean(), 2
        ),
        "copied_percent": round(
            100 * frame["copied_input"].mean(), 2
        ),
        "catastrophic_percent": round(
            100 * frame["catastrophic_candidate"].mean(), 2
        ),
        "maximum_harm_delta": int(frame["harm_delta"].max()),
        "positive_harm_p95": round(
            positive_harm.quantile(0.95), 2
        ) if len(positive_harm) else 0.0,
    }


group_summary = pd.DataFrame([
    {
        "evaluation_group": group_name,
        **summarize(group),
    }
    for group_name, group
    in df.groupby("evaluation_group", dropna=False)
])

synthetic = df[
    df["evaluation_group"] == "synthetic_error"
]

error_summary = pd.DataFrame([
    {
        "error_type": error_type,
        **summarize(group),
    }
    for error_type, group
    in synthetic.groupby("error_type", dropna=False)
]).sort_values(
    ["exact_percent", "harmed_percent"],
    ascending=[True, False],
)

print("\nGROUP SUMMARY")
display(group_summary)

print("\nSYNTHETIC RESULTS BY ERROR TYPE")
display(error_summary)


# --------------------------------------------------
# 4. Show important and most harmful examples
# --------------------------------------------------

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

display_columns = [
    "row_id",
    "evaluation_group",
    "error_type",
    "corrupted_text",
    "clean_text",
    prediction_column,
    "input_distance",
    "output_distance",
    "harm_delta",
    "model_edit_distance",
    "length_ratio",
    "normalized_edit_ratio",
    "catastrophic_candidate",
]

priority_ids = [180, 393, 230, 66]

priority_rows = df[df["row_id"].isin(priority_ids)].copy()

worst_rows = (
    df.sort_values(
        ["harm_delta", "model_edit_distance"],
        ascending=[False, False],
    )
    .head(20)
    .copy()
)

print("\nPREVIOUSLY IDENTIFIED PRIORITY ROWS")
display(priority_rows[display_columns])

print("\n20 MOST HARMFUL V1 OUTPUTS")
display(worst_rows[display_columns])


# --------------------------------------------------
# 5. Create a human-review CSV without overwriting
# --------------------------------------------------

weak_types = {
    "hyphen_to_space",
    "kazakh_letter_loss",
    "missing_space",
    "person_agreement_heuristic",
    "suffix_harmony_heuristic",
}

weak_parts = []

for error_type, group in df[
    df["error_type"].isin(weak_types)
].groupby("error_type"):
    weak_parts.append(
        group.sample(
            n=min(25, len(group)),
            random_state=42,
        )
    )

weak_sample = (
    pd.concat(weak_parts, ignore_index=True)
    if weak_parts
    else df.iloc[0:0].copy()
)

audit_df = (
    pd.concat(
        [
            priority_rows,
            worst_rows,
            weak_sample,
        ],
        ignore_index=True,
    )
    .drop_duplicates("row_id")
    .sort_values(
        ["catastrophic_candidate", "harm_delta"],
        ascending=[False, False],
    )
)

audit_df["human_pair_valid"] = ""
audit_df["failure_cause"] = ""
audit_df["review_notes"] = ""

export_columns = display_columns + [
    "human_pair_valid",
    "failure_cause",
    "review_notes",
]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
audit_path = (
    EVALUATION_DIR
    / f"v1_audit_candidates_{timestamp}.csv"
)

audit_df[export_columns].to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig",
)

print("\nAudit file saved safely to:")
print(audit_path)
print("Rows selected for human review:", len(audit_df))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 14.4 MB/s eta 0:00:00
Mounted at /content/drive
Loaded: /content/drive/MyDrive/kazakh-gec/evaluation/predictions_with_extracted_content.parquet
Examples: 400
Prediction column: after_content

GROUP SUMMARY


,evaluation_group,examples,exact_percent,cer_percent,improved_percent,harmed_percent,copied_percent,catastrophic_percent,maximum_harm_delta,positive_harm_p95
0,clean_control,100,91.00,0.69,0.0,9.0,91.0,2.0,46,32.0
1,synthetic_error,300,63.67,1.12,65.0,9.0,23.0,1.0,111,29.2



SYNTHETIC RESULTS BY ERROR TYPE


,error_type,examples,exact_percent,cer_percent,improved_percent,harmed_percent,copied_percent,catastrophic_percent,maximum_harm_delta,positive_harm_p95
8,typo,37,13.51,1.89,21.62,27.03,40.54,0.00,6,5.10
1,hyphen_to_space,38,50.00,2.31,50.00,5.26,44.74,2.63,111,105.55
3,kazakh_letter_loss,38,52.63,0.70,52.63,10.53,36.84,0.00,2,1.85
5,missing_space,45,55.56,0.75,55.56,13.33,28.89,0.00,4,3.50
6,person_agreement_heuristic,11,63.64,1.44,63.64,9.09,18.18,0.00,1,1.00
7,suffix_harmony_heuristic,37,75.68,1.11,78.38,5.41,13.51,2.70,31,29.50
4,missing_final_punctuation,26,88.46,0.16,88.46,3.85,3.85,0.00,1,1.00
0,double_space,34,94.12,0.83,94.12,2.94,0.00,2.94,25,25.00
2,initial_case,34,94.12,0.06,94.12,0.00,5.88,0.00,0,0.00



PREVIOUSLY IDENTIFIED PRIORITY ROWS


,row_id,evaluation_group,error_type,corrupted_text,clean_text,after_content,input_distance,output_distance,harm_delta,model_edit_distance,length_ratio,normalized_edit_ratio,catastrophic_candidate
66,66,synthetic_error,double_space,"Құбыржолдардың зақымдалу ықтималдығын болғызбау үшін оларды төсеудің кез келген түрінде су асты өткелдерінің бойында — өткелдердің шеткі сызықтарының осінен әрбір жағынан жүз метр қашықтықтағы, параллель жазықтар арасында орналасқан, су бетінен түбіне дейінгі су кеңістігінің учаскесі түрінде күзет аймақтары белгіленеді.","Құбыржолдардың зақымдалу ықтималдығын болғызбау үшін оларды төсеудің кез келген түрінде су асты өткелдерінің бойында — өткелдердің шеткі сызықтарының осінен әрбір жағынан жүз метр қашықтықтағы, параллель жазықтар арасында орналасқан, су бетінен түбіне дейінгі су кеңістігінің учаскесі түрінде күзет аймақтары белгіленеді.","Құбыржолдардың зақымдалу ықтималдығын болғызбау үшін оларды төсеудің кез-келген түрінде су асты өткелдерінің бойында — өткелдердің шеткі сызықтарының осінен әрбір жағынан жүз метр қашықтықтағы, параллель жазықтар арасында орналасқан, су бетінен түбіне дейінгі су кеңістігінің учаскесі түрінде күз",1,26,25,27,0.919255,0.083851,True
180,180,synthetic_error,hyphen_to_space,"Кондоминиум объектісін басқаруға және кондоминиум объектісінің ортақ мүлкін күтіп ұстауға арналған шығыстардың мөлшері және оларды көппәтерлі тұрғын үйдің пәтерлері, тұрғын емес үй-жайлары меншік иелерінің төлеу тәртібі ""Тұрғын үй қатынастары туралы"" Қазақстан Республикасының Заңында айқындалған тәртіппен көппәтерлі тұрғын үйдің пәтерлері, тұрғын емес үй-жайлары меншік иелерінің жиналысында бекітіледі.","Кондоминиум объектісін басқаруға және кондоминиум объектісінің ортақ мүлкін күтіп-ұстауға арналған шығыстардың мөлшері және оларды көппәтерлі тұрғын үйдің пәтерлері, тұрғын емес үй-жайлары меншік иелерінің төлеу тәртібі ""Тұрғын үй қатынастары туралы"" Қазақстан Республикасының Заңында айқындалған тәртіппен көппәтерлі тұрғын үйдің пәтерлері, тұрғын емес үй-жайлары меншік иелерінің жиналысында бекітіледі.","Кондоминиум объектісін басқаруға және кондоминиум объектісінің ортақ мүлкін күтіп ұстауға арналған шығыстардың мөлшері және оларды көппәтерлі тұрғын үйдің пәтерлері, тұрғын емес үй-жайлары меншік иелерінің төлеу тәртібі ""Тұрғын үй қатынастары туралы"" Қазақстан Республикасының Заңында айқындалғ",1,112,111,111,0.725926,0.274074,True
230,230,synthetic_error,suffix_harmony_heuristic,"Қазақстан Республикасының газ және газбен жабдықтау туралы заңнамасынде белгіленген газбен жабдықтау жүйелері объектілерін пайдалану жөніндегі шектеулерді сақтамау шағын кәсіпкерлік субъектілеріне – елу, орта кәсіпкерлік субъектілеріне – бір жүз, ірі кәсіпкерлік субъектілеріне үш жүз айлық есептік көрсеткіш мөлшерінде айыппұл салуға әкеп соғады.","Қазақстан Республикасының газ және газбен жабдықтау туралы заңнамасында белгіленген газбен жабдықтау жүйелері объектілерін пайдалану жөніндегі шектеулерді сақтамау шағын кәсіпкерлік субъектілеріне – елу, орта кәсіпкерлік субъектілеріне – бір жүз, ірі кәсіпкерлік субъектілеріне үш жүз айлық есептік көрсеткіш мөлшерінде айыппұл салуға әкеп соғады.","Қазақстан Республикасының газ және газбен жабдықтау туралы заңнамасында белгіленген газбен жабдықтау жүйелері объектілерін пайдалану жөніндегі шектеулерді сақтамау шағын кәсіпкерлік субъектілеріне – елу, орта кәсіпкерлік субъектілеріне – бір жүз, ірі кәсіпкерлік субъектілеріне үш жүз айлық есептік көрсеткіш мөлшер",1,32,31,33,0.907781,0.095101,True
393,393,clean_control,clean_identity,"Ұсталған немесе қамаудағы тұлғаның ұсталу немесе қамалу уақытындағы жасаған тәртіптік құқық бұзушылық болып табылатын тәртібінің сипаты, тәртіптік жазаның түрі мен ұзақтығы, сонымен қатар осындай жазаны тағайындауға құзыретті билік өкілдері тиісті түрде жарияланған заңдарда немесе заңға сәйкес орнатылған ережелерде нақты анықталуы тиіс.","Ұсталған немесе қамаудағы тұлғаның ұсталу немесе қамалу уақытындағы жасаған тәртіптік құқық бұзушылық болып табы


20 MOST HARMFUL V1 OUTPUTS


,row_id,evaluation_group,error_type,corrupted_text,clean_text,after_content,input_distance,output_distance,harm_delta,model_edit_distance,length_ratio,normalized_edit_ratio,catastrophic_candidate
180,180,synthetic_error,hyphen_to_space,"Кондоминиум объектісін басқаруға және кондоминиум объектісінің ортақ мүлкін күтіп ұстауға арналған шығыстардың мөлшері және оларды көппәтерлі тұрғын үйдің пәтерлері, тұрғын емес үй-жайлары меншік иелерінің төлеу тәртібі ""Тұрғын үй қатынастары туралы"" Қазақстан Республикасының Заңында айқындалған тәртіппен көппәтерлі тұрғын үйдің пәтерлері, тұрғын емес үй-жайлары меншік иелерінің жиналысында бекітіледі.","Кондоминиум объектісін басқаруға және кондоминиум объектісінің ортақ мүлкін күтіп-ұстауға арналған шығыстардың мөлшері және оларды көппәтерлі тұрғын үйдің пәтерлері, тұрғын емес үй-жайлары меншік иелерінің төлеу тәртібі ""Тұрғын үй қатынастары туралы"" Қазақстан Республикасының Заңында айқындалған тәртіппен көппәтерлі тұрғын үйдің пәтерлері, тұрғын емес үй-жайлары меншік иелерінің жиналысында бекітіледі.","Кондоминиум объектісін басқаруға және кондоминиум объектісінің ортақ мүлкін күтіп ұстауға арналған шығыстардың мөлшері және оларды көппәтерлі тұрғын үйдің пәтерлері, тұрғын емес үй-жайлары меншік иелерінің төлеу тәртібі ""Тұрғын үй қатынастары туралы"" Қазақстан Республикасының Заңында айқындалғ",1,112,111,111,0.725926,0.274074,True
393,393,clean_control,clean_identity,"Ұсталған немесе қамаудағы тұлғаның ұсталу немесе қамалу уақытындағы жасаған тәртіптік құқық бұзушылық болып табылатын тәртібінің сипаты, тәртіптік жазаның түрі мен ұзақтығы, сонымен қатар осындай жазаны тағайындауға құзыретті билік өкілдері тиісті түрде жарияланған заңдарда немесе заңға сәйкес орнатылған ережелерде нақты анықталуы тиіс.","Ұсталған немесе қамаудағы тұлғаның ұсталу немесе қамалу уақытындағы жасаған тәртіптік құқық бұзушылық болып табылатын тәртібінің сипаты, тәртіптік жазаның түрі мен ұзақтығы, сонымен қатар осындай жазаны тағайындауға құзыретті билік өкілдері тиісті түрде жарияланған заңдарда немесе заңға сәйкес орнатылған ережелерде нақты анықталуы тиіс.","Ұсталған немесе қамаудағы тұлғаның ұсталу немесе қамалу уақытындағы жасаған тәртіптік құқық бұзушылық болып табылатын тәртібінің сипаты, тәртіптік жазаның түрі мен ұзақтығы, сонымен қатар осындай жазаны тағайындауға құзыретті билік өкілдері тиісті түрде жарияланған заңдарда немесе заңға сәйк",0,46,46,46,0.863905,0.136095,True
230,230,synthetic_error,suffix_harmony_heuristic,"Қазақстан Республикасының газ және газбен жабдықтау туралы заңнамасынде белгіленген газбен жабдықтау жүйелері объектілерін пайдалану жөніндегі шектеулерді сақтамау шағын кәсіпкерлік субъектілеріне – елу, орта кәсіпкерлік субъектілеріне – бір жүз, ірі кәсіпкерлік субъектілеріне үш жүз айлық есептік көрсеткіш мөлшерінде айыппұл салуға әкеп соғады.","Қазақстан Республикасының газ және газбен жабдықтау туралы заңнамасында белгіленген газбен жабдықтау жүйелері объектілерін пайдалану жөніндегі шектеулерді сақтамау шағын кәсіпкерлік субъектілеріне – елу, орта кәсіпкерлік субъектілеріне – бір жүз, ірі кәсіпкерлік субъектілеріне үш жүз айлық есептік көрсеткіш мөлшерінде айыппұл салуға әкеп соғады.","Қазақстан Республикасының газ және газбен жабдықтау туралы заңнамасында белгіленген газбен жабдықтау жүйелері объектілерін пайдалану жөніндегі шектеулерді сақтамау шағын кәсіпкерлік субъектілеріне – елу, орта кәсіпкерлік субъектілеріне – бір жүз, ірі кәсіпкерлік субъектілеріне үш жүз айлық есептік көрсеткіш мөлшер",1,32,31,33,0.907781,0.095101,True
66,66,synthetic_error,double_space,"Құбыржолдардың зақымдалу ықтималдығын болғызбау үшін оларды төсеудің кез келген түрінде су асты өткелдерінің бойында — өткелдердің шеткі сызықтарының осінен әрбір жағынан жүз метр қашықтықтағы, параллель жазықтар арасында орналасқан, су бетінен түбіне дейінгі су кеңістігінің учаскесі түрінде күзет аймақтары белгіленеді.","Құбыржолдардың зақымдалу ықтималдығын болғызбау үшін оларды төсеудің кез келген түрінде су асты өтк


Audit file saved safely to:
/content/drive/MyDrive/kazakh-gec/evaluation/v1_audit_candidates_20260811_135639.csv
Rows selected for human review: 127


In [ ]:
# STEP 2 — DIAGNOSE INPUT/OUTPUT TOKEN TRUNCATION
# No GPU or model weights are required.

from datetime import datetime
from pathlib import Path

import pandas as pd
from google.colab import drive
from transformers import AutoTokenizer
from IPython.display import display


# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/kazakh-gec")

ADAPTER_DIR = (
    PROJECT_DIR
    / "models"
    / "qwen3-0.6b-kazakh-gec"
    / "final"
)

EVALUATION_DIR = PROJECT_DIR / "evaluation"

prediction_files = [
    EVALUATION_DIR / "predictions_with_extracted_content.parquet",
    EVALUATION_DIR / "before_after_predictions.parquet",
]

prediction_path = next(
    (path for path in prediction_files if path.exists()),
    None,
)

if prediction_path is None:
    raise FileNotFoundError(
        f"No prediction file found in {EVALUATION_DIR}"
    )

if not (ADAPTER_DIR / "tokenizer_config.json").exists():
    raise FileNotFoundError(
        f"Tokenizer was not found in {ADAPTER_DIR}"
    )


# --------------------------------------------------
# 2. Load predictions and tokenizer
# --------------------------------------------------

df = pd.read_parquet(prediction_path).copy()

if "row_id" not in df.columns:
    df.insert(0, "row_id", range(len(df)))

raw_column = (
    "after_training"
    if "after_training" in df.columns
    else "after_content"
)

content_column = (
    "after_content"
    if "after_content" in df.columns
    else raw_column
)

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))

SYSTEM_PROMPT = (
    "Сен қазақ тіліндегі мәтінді түзететін редакторсың. "
    "Мағынасын өзгертпе. "
    "Жауапта тек түзетілген мәтінді бер, түсіндірме қоспа."
)

USER_PREFIX = "Қазақша мәтінді түзет:\n"

OLD_MAX_INPUT_TOKENS = 384
OLD_MAX_NEW_TOKENS = 192


def make_prompt_text(source_text):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": USER_PREFIX + str(source_text),
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def token_count(text):
    return len(
        tokenizer(
            str(text),
            add_special_tokens=False,
        )["input_ids"]
    )


# --------------------------------------------------
# 3. Calculate token-budget diagnostics
# --------------------------------------------------

df["_full_prompt"] = df["corrupted_text"].map(make_prompt_text)

df["source_tokens"] = df["corrupted_text"].map(token_count)
df["full_prompt_tokens"] = df["_full_prompt"].map(token_count)
df["raw_output_tokens"] = df[raw_column].map(token_count)
df["content_output_tokens"] = df[content_column].map(token_count)

df["prompt_exceeded_384"] = (
    df["full_prompt_tokens"] > OLD_MAX_INPUT_TOKENS
)

# Tokenizing decoded text is only a proxy for the original generated count.
# Values near 192 are strong evidence that generation hit the old cap.
df["output_near_192_cap"] = (
    df["raw_output_tokens"] >= OLD_MAX_NEW_TOKENS - 4
)

df["raw_equals_content"] = (
    df[raw_column].astype(str).str.strip()
    == df[content_column].astype(str).str.strip()
)

df["output_shorter_than_source"] = (
    df[content_column].astype(str).str.len()
    < df["corrupted_text"].astype(str).str.len()
)

df["output_ends_midword"] = (
    df[content_column]
    .astype(str)
    .str.rstrip()
    .str[-1:]
    .map(lambda value: value.isalpha())
)

df["raw_tail"] = (
    df[raw_column]
    .astype(str)
    .map(lambda value: value[-100:])
)

df["content_tail"] = (
    df[content_column]
    .astype(str)
    .map(lambda value: value[-100:])
)


# --------------------------------------------------
# 4. Inspect the known failures
# --------------------------------------------------

diagnostic_ids = [66, 180, 230, 337, 393]

diagnostic_columns = [
    "row_id",
    "evaluation_group",
    "error_type",
    "source_tokens",
    "full_prompt_tokens",
    "raw_output_tokens",
    "content_output_tokens",
    "prompt_exceeded_384",
    "output_near_192_cap",
    "raw_equals_content",
    "output_shorter_than_source",
    "output_ends_midword",
    "raw_tail",
    "content_tail",
]

diagnostic_df = df[
    df["row_id"].isin(diagnostic_ids)
][diagnostic_columns].copy()

pd.set_option("display.max_colwidth", None)

print("TOKEN-BUDGET DIAGNOSTIC")
display(diagnostic_df)


# --------------------------------------------------
# 5. Save without overwriting anything
# --------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

diagnostic_path = (
    EVALUATION_DIR
    / f"v1_token_budget_diagnostic_{timestamp}.csv"
)

diagnostic_df.to_csv(
    diagnostic_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved to:", diagnostic_path)

TOKEN-BUDGET DIAGNOSTIC


,row_id,evaluation_group,error_type,source_tokens,full_prompt_tokens,raw_output_tokens,content_output_tokens,prompt_exceeded_384,output_near_192_cap,raw_equals_content,output_shorter_than_source,output_ends_midword,raw_tail,content_tail
66,66,synthetic_error,double_space,206,319,192,192,False,True,True,True,True,"раллель жазықтар арасында орналасқан, су бетінен түбіне дейінгі су кеңістігінің учаскесі түрінде күз","раллель жазықтар арасында орналасқан, су бетінен түбіне дейінгі су кеңістігінің учаскесі түрінде күз"
180,180,synthetic_error,hyphen_to_space,267,380,192,192,False,True,True,True,True,"к иелерінің төлеу тәртібі ""Тұрғын үй қатынастары туралы"" Қазақстан Республикасының Заңында айқындалғ","к иелерінің төлеу тәртібі ""Тұрғын үй қатынастары туралы"" Қазақстан Республикасының Заңында айқындалғ"
230,230,synthetic_error,suffix_harmony_heuristic,215,328,192,192,False,True,True,True,True,"ерлік субъектілеріне – бір жүз, ірі кәсіпкерлік субъектілеріне үш жүз айлық есептік көрсеткіш мөлшер","ерлік субъектілеріне – бір жүз, ірі кәсіпкерлік субъектілеріне үш жүз айлық есептік көрсеткіш мөлшер"
337,337,clean_control,clean_identity,199,312,192,192,False,True,True,True,True,"тергеулерге, қылмыстық қудалауға немесе тапсырудың рәсімдеріне байланысты бір-біріне мейлінше көмек","тергеулерге, қылмыстық қудалауға немесе тапсырудың рәсімдеріне байланысты бір-біріне мейлінше көмек"
393,393,clean_control,clean_identity,220,333,192,192,False,True,True,True,True,дай жазаны тағайындауға құзыретті билік өкілдері тиісті түрде жарияланған заңдарда немесе заңға сәйк,дай жазаны тағайындауға құзыретті билік өкілдері тиісті түрде жарияланған заңдарда немесе заңға сәйк


Saved to: /content/drive/MyDrive/kazakh-gec/evaluation/v1_token_budget_diagnostic_20260811_142402.csv


In [ ]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
%pip install -q -U "transformers>=4.51.0" "peft>=0.18.0" accelerate rapidfuzz pyarrow

from peft import PeftModel
print("PEFT is ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.3 MB/s eta 0:00:00
PEFT is ready


In [ ]:
# STEP 3 — REGENERATE ONLY TOKEN-CAPPED OUTPUTS
# Preserves all old results and saves new timestamped files.

from datetime import datetime
from pathlib import Path
import gc
import unicodedata

import pandas as pd
import torch
from google.colab import drive
from peft import PeftModel
from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from IPython.display import display


# --------------------------------------------------
# 1. Configuration and paths
# --------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

MODEL_ID = "Qwen/Qwen3-0.6B"

PROJECT_DIR = Path("/content/drive/MyDrive/kazakh-gec")

ADAPTER_DIR = (
    PROJECT_DIR
    / "models"
    / "qwen3-0.6b-kazakh-gec"
    / "final"
)

EVALUATION_DIR = PROJECT_DIR / "evaluation"

prediction_files = [
    EVALUATION_DIR / "predictions_with_extracted_content.parquet",
    EVALUATION_DIR / "before_after_predictions.parquet",
]

prediction_path = next(
    (path for path in prediction_files if path.exists()),
    None,
)

if prediction_path is None:
    raise FileNotFoundError(
        f"No prediction file found in {EVALUATION_DIR}"
    )

assert (ADAPTER_DIR / "adapter_config.json").exists()
assert (ADAPTER_DIR / "tokenizer_config.json").exists()

OLD_MAX_NEW_TOKENS = 192
DYNAMIC_MARGIN = 64
HARD_MAX_NEW_TOKENS = 512

SYSTEM_PROMPT = (
    "Сен қазақ тіліндегі мәтінді түзететін редакторсың. "
    "Мағынасын өзгертпе. "
    "Жауапта тек түзетілген мәтінді бер, түсіндірме қоспа."
)

USER_PREFIX = "Қазақша мәтінді түзет:\n"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


# --------------------------------------------------
# 2. Load predictions and tokenizer
# --------------------------------------------------

df = pd.read_parquet(prediction_path).copy()

if "row_id" not in df.columns:
    df.insert(0, "row_id", range(len(df)))

old_raw_column = (
    "after_training"
    if "after_training" in df.columns
    else "after_content"
)

old_content_column = (
    "after_content"
    if "after_content" in df.columns
    else old_raw_column
)

required_columns = {
    "corrupted_text",
    "clean_text",
    "evaluation_group",
    old_content_column,
}

missing = required_columns - set(df.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

if "error_type" not in df.columns:
    df["error_type"] = "unknown"

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))
tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def make_prompt_text(source_text):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": USER_PREFIX + str(source_text),
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def token_count(text):
    return len(
        tokenizer(
            str(text),
            add_special_tokens=False,
        )["input_ids"]
    )


df["after_content_old_192"] = (
    df[old_content_column].fillna("").astype(str)
)

df["old_raw_output_tokens"] = (
    df[old_raw_column].fillna("").astype(str).map(token_count)
)

# Decoded outputs at or very near 192 tokens are regenerated.
cap_mask = (
    df["old_raw_output_tokens"]
    >= OLD_MAX_NEW_TOKENS - 2
)

cap_indices = df.index[cap_mask].tolist()
cap_row_ids = df.loc[cap_mask, "row_id"].tolist()

print("Prediction file:", prediction_path)
print("Examples reaching the old cap:", len(cap_indices))
print("Row IDs:", cap_row_ids)

if not cap_indices:
    raise ValueError(
        "No capped outputs were detected. Stop here and send this message."
    )


# --------------------------------------------------
# 3. Release any previous model objects
# --------------------------------------------------

for variable_name in [
    "trainer",
    "training_model",
    "trained_model",
    "base_model",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()


# --------------------------------------------------
# 4. Load base model and existing v1 adapter
# --------------------------------------------------

# CPU model loading

print("Running on CPU. Only capped rows will be regenerated.")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
)

trained_model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
)

trained_model.to("cpu")
trained_model.eval()
trained_model.config.use_cache = True


# --------------------------------------------------
# 5. Dynamic-budget generation
# --------------------------------------------------

def generate_with_dynamic_budget(
    model,
    source_texts,
    batch_size=4,
):
    predictions = []
    generated_token_counts = []
    token_budgets = []
    hit_caps = []
    prompt_token_counts = []

    model_device = next(model.parameters()).device

    for start in tqdm(
        range(0, len(source_texts), batch_size),
        desc="Repairing capped outputs",
    ):
        batch_sources = source_texts[
            start : start + batch_size
        ]

        batch_prompts = [
            make_prompt_text(source)
            for source in batch_sources
        ]

        source_token_counts = [
            token_count(source)
            for source in batch_sources
        ]

        batch_budget = min(
            HARD_MAX_NEW_TOKENS,
            max(source_token_counts) + DYNAMIC_MARGIN,
        )

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=False,
        ).to(model_device)

        batch_prompt_tokens = inputs["input_ids"].shape[1]

        # This experiment should remain comfortably below
        # the model's context limit.
        if batch_prompt_tokens + batch_budget > 1024:
            raise ValueError(
                "Prompt plus output budget exceeded 1024 tokens. "
                "Run the affected examples individually."
            )

        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=batch_budget,
                do_sample=False,
                num_beams=1,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated_only = generated[
            :,
            inputs["input_ids"].shape[1] :,
        ]

        decoded = tokenizer.batch_decode(
            generated_only,
            skip_special_tokens=True,
        )

        for token_ids, decoded_text in zip(
            generated_only.tolist(),
            decoded,
        ):
            try:
                eos_position = token_ids.index(
                    tokenizer.eos_token_id
                )
            except ValueError:
                eos_position = None

            if eos_position is None:
                generated_count = len(token_ids)
                hit_cap = generated_count >= batch_budget
            else:
                generated_count = eos_position
                hit_cap = False

            predictions.append(decoded_text.strip())
            generated_token_counts.append(generated_count)
            token_budgets.append(batch_budget)
            hit_caps.append(hit_cap)
            prompt_token_counts.append(batch_prompt_tokens)

    return {
        "predictions": predictions,
        "generated_token_counts": generated_token_counts,
        "token_budgets": token_budgets,
        "hit_caps": hit_caps,
        "prompt_token_counts": prompt_token_counts,
    }


sources_to_repair = (
    df.loc[cap_indices, "corrupted_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

repair_result = generate_with_dynamic_budget(
    trained_model,
    sources_to_repair,
)


# --------------------------------------------------
# 6. Insert repaired outputs without deleting old ones
# --------------------------------------------------

df["after_content_dynamic"] = df["after_content_old_192"]

df["dynamic_generated_tokens"] = pd.NA
df["dynamic_token_budget"] = pd.NA
df["dynamic_hit_cap"] = pd.NA
df["dynamic_prompt_tokens"] = pd.NA
df["was_regenerated"] = False

df.loc[
    cap_indices,
    "after_content_dynamic",
] = repair_result["predictions"]

df.loc[
    cap_indices,
    "dynamic_generated_tokens",
] = repair_result["generated_token_counts"]

df.loc[
    cap_indices,
    "dynamic_token_budget",
] = repair_result["token_budgets"]

df.loc[
    cap_indices,
    "dynamic_hit_cap",
] = repair_result["hit_caps"]

df.loc[
    cap_indices,
    "dynamic_prompt_tokens",
] = repair_result["prompt_token_counts"]

df.loc[
    cap_indices,
    "was_regenerated",
] = True


# --------------------------------------------------
# 7. Recalculate metrics
# --------------------------------------------------

def normalize_for_evaluation(text):
    return unicodedata.normalize(
        "NFC",
        str(text),
    ).strip()


def calculate_metrics(frame, prediction_column):
    references = [
        normalize_for_evaluation(value)
        for value in frame["clean_text"]
    ]

    sources = [
        normalize_for_evaluation(value)
        for value in frame["corrupted_text"]
    ]

    predictions = [
        normalize_for_evaluation(value)
        for value in frame[prediction_column]
    ]

    input_distances = [
        Levenshtein.distance(reference, source)
        for reference, source in zip(references, sources)
    ]

    output_distances = [
        Levenshtein.distance(reference, prediction)
        for reference, prediction in zip(
            references,
            predictions,
        )
    ]

    total_reference_characters = sum(
        max(len(reference), 1)
        for reference in references
    )

    count = len(frame)

    return {
        "examples": count,
        "exact_percent": round(
            100
            * sum(
                prediction == reference
                for prediction, reference
                in zip(predictions, references)
            )
            / count,
            2,
        ),
        "cer_percent": round(
            100
            * sum(output_distances)
            / total_reference_characters,
            2,
        ),
        "improved_percent": round(
            100
            * sum(
                output_distance < input_distance
                for output_distance, input_distance
                in zip(output_distances, input_distances)
            )
            / count,
            2,
        ),
        "harmed_percent": round(
            100
            * sum(
                output_distance > input_distance
                for output_distance, input_distance
                in zip(output_distances, input_distances)
            )
            / count,
            2,
        ),
        "copied_percent": round(
            100
            * sum(
                prediction == source
                for prediction, source
                in zip(predictions, sources)
            )
            / count,
            2,
        ),
        "maximum_harm_delta": max(
            output_distance - input_distance
            for output_distance, input_distance
            in zip(output_distances, input_distances)
        ),
    }


stages = {
    "copy": "corrupted_text",
    "old_192": "after_content_old_192",
    "dynamic_budget": "after_content_dynamic",
}

group_rows = []

for group_name, group in df.groupby("evaluation_group"):
    for stage, prediction_column in stages.items():
        group_rows.append({
            "evaluation_group": group_name,
            "stage": stage,
            **calculate_metrics(group, prediction_column),
        })

group_metrics = pd.DataFrame(group_rows)

per_error_rows = []

synthetic_df = df[
    df["evaluation_group"] == "synthetic_error"
]

for error_type, group in synthetic_df.groupby("error_type"):
    for stage in ["old_192", "dynamic_budget"]:
        per_error_rows.append({
            "error_type": error_type,
            "stage": stage,
            **calculate_metrics(group, stages[stage]),
        })

per_error_metrics = pd.DataFrame(per_error_rows)


# --------------------------------------------------
# 8. Inspect repaired cases
# --------------------------------------------------

comparison_columns = [
    "row_id",
    "evaluation_group",
    "error_type",
    "corrupted_text",
    "clean_text",
    "after_content_old_192",
    "after_content_dynamic",
    "old_raw_output_tokens",
    "dynamic_generated_tokens",
    "dynamic_token_budget",
    "dynamic_hit_cap",
]

print("\nREPAIRED OUTPUTS")
display(df.loc[cap_indices, comparison_columns])

print("\nGROUP METRICS: OLD VS DYNAMIC BUDGET")
display(group_metrics)

print("\nPER-ERROR METRICS")
display(per_error_metrics)


# --------------------------------------------------
# 9. Save new artifacts without overwriting v1
# --------------------------------------------------

prediction_output_path = (
    EVALUATION_DIR
    / f"v1_dynamic_budget_predictions_{timestamp}.parquet"
)

group_metrics_path = (
    EVALUATION_DIR
    / f"v1_dynamic_budget_group_metrics_{timestamp}.csv"
)

per_error_metrics_path = (
    EVALUATION_DIR
    / f"v1_dynamic_budget_per_error_metrics_{timestamp}.csv"
)

df.to_parquet(
    prediction_output_path,
    index=False,
)

group_metrics.to_csv(
    group_metrics_path,
    index=False,
)

per_error_metrics.to_csv(
    per_error_metrics_path,
    index=False,
)

print("\nSaved predictions:", prediction_output_path)
print("Saved group metrics:", group_metrics_path)
print("Saved per-error metrics:", per_error_metrics_path)

if any(repair_result["hit_caps"]):
    print(
        "\nWARNING: At least one repaired output still hit "
        "the dynamic token cap."
    )
else:
    print(
        "\nSuccess: no repaired output reached "
        "the dynamic token cap."
    )

Prediction file: /content/drive/MyDrive/kazakh-gec/evaluation/predictions_with_extracted_content.parquet
Examples reaching the old cap: 5
Row IDs: [66, 180, 230, 337, 393]
Running on CPU. Only capped rows will be regenerated.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Repairing capped outputs:   0%|          | 0/2 [00:00<?, ?it/s]


REPAIRED OUTPUTS


...


GROUP METRICS: OLD VS DYNAMIC BUDGET


,evaluation_group,stage,examples,exact_percent,cer_percent,improved_percent,harmed_percent,copied_percent,maximum_harm_delta
0,clean_control,copy,100,100.00,0.00,0.00,0.0,100.00,0
1,clean_control,old_192,100,91.00,0.69,0.00,9.0,91.00,46
2,clean_control,dynamic_budget,100,93.00,0.14,0.00,7.0,93.00,5
3,synthetic_error,copy,300,0.00,1.12,0.00,0.0,100.00,0
4,synthetic_error,old_192,300,63.67,1.12,65.00,9.0,23.00,111
5,synthetic_error,dynamic_budget,300,64.00,0.55,65.33,8.0,23.33,6



PER-ERROR METRICS


,error_type,stage,examples,exact_percent,cer_percent,improved_percent,harmed_percent,copied_percent,maximum_harm_delta
0,double_space,old_192,34,94.12,0.83,94.12,2.94,0.00,25
1,double_space,dynamic_budget,34,94.12,0.06,94.12,0.00,0.00,0
2,hyphen_to_space,old_192,38,50.00,2.31,50.00,5.26,44.74,111
3,hyphen_to_space,dynamic_budget,38,50.00,0.37,50.00,2.63,47.37,2
4,initial_case,old_192,34,94.12,0.06,94.12,0.00,5.88,0
5,initial_case,dynamic_budget,34,94.12,0.06,94.12,0.00,5.88,0
6,kazakh_letter_loss,old_192,38,52.63,0.70,52.63,10.53,36.84,2
7,kazakh_letter_loss,dynamic_budget,38,52.63,0.70,52.63,10.53,36.84,2
8,missing_final_punctuation,old_192,26,88.46,0.16,88.46,3.85,3.85,1
9,missing_final_punctuation,dynamic_budget,26,88.46,0.16,88.46,3.85,3.85,1



Saved predictions: /content/drive/MyDrive/kazakh-gec/evaluation/v1_dynamic_budget_predictions_20260811_144916.parquet
Saved group metrics: /content/drive/MyDrive/kazakh-gec/evaluation/v1_dynamic_budget_group_metrics_20260811_144916.csv
Saved per-error metrics: /content/drive/MyDrive/kazakh-gec/evaluation/v1_dynamic_budget_per_error_metrics_20260811_144916.csv

Success: no repaired output reached the dynamic token cap.


In [ ]:
# STEP 4 — FIX PARSER, APPLY SAFETY GUARD, RECOMPUTE METRICS
# CPU only; does not load or run the model.

%pip install -q rapidfuzz pyarrow

from datetime import datetime
from pathlib import Path
import re
import unicodedata

import pandas as pd
from google.colab import drive, files
from IPython.display import display
from rapidfuzz.distance import Levenshtein


# --------------------------------------------------
# 1. Locate the latest dynamic-budget predictions
# --------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/kazakh-gec")
EVALUATION_DIR = PROJECT_DIR / "evaluation"

prediction_files = sorted(
    EVALUATION_DIR.glob(
        "v1_dynamic_budget_predictions_*.parquet"
    ),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not prediction_files:
    raise FileNotFoundError(
        f"No dynamic-budget prediction file found in {EVALUATION_DIR}"
    )

prediction_path = prediction_files[0]
df = pd.read_parquet(prediction_path).copy()

print("Loaded:", prediction_path)
print("Rows:", len(df))


# --------------------------------------------------
# 2. Safe normalization and extraction
# --------------------------------------------------

def normalize_text(value):
    if pd.isna(value):
        return ""

    return unicodedata.normalize(
        "NFC",
        str(value),
    ).strip()


# IMPORTANT: "Жауап" is removed only when it is an
# explicit response label followed by a colon.
LABEL_PATTERN = re.compile(
    r"^\s*(?:Жауап|Түзетілген\s+мәтін)\s*:\s*",
    flags=re.IGNORECASE,
)


def safe_extract(raw_output):
    text = normalize_text(raw_output)
    return LABEL_PATTERN.sub("", text, count=1).strip()


if "was_regenerated" in df.columns:
    regenerated = df["was_regenerated"].fillna(False)

    if regenerated.dtype == object:
        regenerated = (
            regenerated.astype(str)
            .str.lower()
            .eq("true")
        )
    else:
        regenerated = regenerated.astype(bool)
else:
    regenerated = pd.Series(False, index=df.index)


# Start with the dynamic outputs.
df["parser_fixed_output"] = (
    df["after_content_dynamic"]
    .map(normalize_text)
)

# For rows not regenerated in Step 3, recover the raw
# model output and apply the corrected parser.
if "after_training" in df.columns:
    non_regenerated = ~regenerated

    df.loc[
        non_regenerated,
        "parser_fixed_output",
    ] = (
        df.loc[non_regenerated, "after_training"]
        .map(safe_extract)
    )


# --------------------------------------------------
# 3. Verify the parser repair
# --------------------------------------------------

original_dynamic = (
    df["after_content_dynamic"]
    .map(normalize_text)
)

parser_changed = (
    df["parser_fixed_output"] != original_dynamic
)

parser_changed_ids = (
    df.loc[parser_changed, "row_id"]
    .astype(int)
    .tolist()
)

print("\nParser-changed row IDs:", parser_changed_ids)
print("Expected: [255]")

display(
    df.loc[
        df["row_id"] == 255,
        [
            "row_id",
            "corrupted_text",
            "clean_text",
            "after_training",
            "after_content_dynamic",
            "parser_fixed_output",
        ],
    ]
)


# --------------------------------------------------
# 4. Calculate guardrail signals
# --------------------------------------------------

sources = df["corrupted_text"].map(normalize_text)
references = df["clean_text"].map(normalize_text)
predictions = df["parser_fixed_output"].map(normalize_text)

df["input_distance_recomputed"] = [
    Levenshtein.distance(reference, source)
    for reference, source in zip(references, sources)
]

df["parser_fixed_distance"] = [
    Levenshtein.distance(reference, prediction)
    for reference, prediction in zip(
        references,
        predictions,
    )
]

df["parser_fixed_harm_delta"] = (
    df["parser_fixed_distance"]
    - df["input_distance_recomputed"]
)

df["model_edit_distance"] = [
    Levenshtein.distance(source, prediction)
    for source, prediction in zip(
        sources,
        predictions,
    )
]

df["model_edit_ratio"] = [
    distance / max(len(source), len(prediction), 1)
    for distance, source, prediction in zip(
        df["model_edit_distance"],
        sources,
        predictions,
    )
]


# Exploratory conservative copy-backoff.
df["guard_triggered"] = (
    (df["model_edit_distance"] >= 3)
    & (df["model_edit_ratio"] >= 0.01)
)

df["final_output"] = df["parser_fixed_output"]

df.loc[
    df["guard_triggered"],
    "final_output",
] = sources[df["guard_triggered"]]

df["guard_reason"] = "accepted"

df.loc[
    df["guard_triggered"],
    "guard_reason",
] = "copy_backoff_large_edit"


# --------------------------------------------------
# 5. Recalculate final distances
# --------------------------------------------------

final_predictions = (
    df["final_output"]
    .map(normalize_text)
)

df["final_content_distance"] = [
    Levenshtein.distance(reference, prediction)
    for reference, prediction in zip(
        references,
        final_predictions,
    )
]

df["final_harm_delta"] = (
    df["final_content_distance"]
    - df["input_distance_recomputed"]
)

guarded_ids = (
    df.loc[df["guard_triggered"], "row_id"]
    .astype(int)
    .tolist()
)

print("\nGuard-triggered row IDs:", guarded_ids)
print(
    "Expected:",
    [1, 37, 53, 121, 143, 160, 191, 240, 335, 342],
)


# --------------------------------------------------
# 6. Metrics
# --------------------------------------------------

def calculate_metrics(frame, prediction_column):
    refs = (
        frame["clean_text"]
        .map(normalize_text)
        .tolist()
    )

    inputs = (
        frame["corrupted_text"]
        .map(normalize_text)
        .tolist()
    )

    outputs = (
        frame[prediction_column]
        .map(normalize_text)
        .tolist()
    )

    input_distances = [
        Levenshtein.distance(reference, source)
        for reference, source in zip(refs, inputs)
    ]

    output_distances = [
        Levenshtein.distance(reference, output)
        for reference, output in zip(refs, outputs)
    ]

    harm_deltas = [
        output_distance - input_distance
        for output_distance, input_distance
        in zip(output_distances, input_distances)
    ]

    count = len(frame)
    reference_characters = sum(
        max(len(reference), 1)
        for reference in refs
    )

    return {
        "examples": count,
        "exact_percent": round(
            100
            * sum(o == r for o, r in zip(outputs, refs))
            / count,
            2,
        ),
        "cer_percent": round(
            100
            * sum(output_distances)
            / reference_characters,
            4,
        ),
        "improved_percent": round(
            100
            * sum(
                output < input_
                for output, input_
                in zip(output_distances, input_distances)
            )
            / count,
            2,
        ),
        "harmed_percent": round(
            100
            * sum(
                output > input_
                for output, input_
                in zip(output_distances, input_distances)
            )
            / count,
            2,
        ),
        "copied_percent": round(
            100
            * sum(o == i for o, i in zip(outputs, inputs))
            / count,
            2,
        ),
        "catastrophic_percent": round(
            100
            * sum(delta > 10 for delta in harm_deltas)
            / count,
            2,
        ),
        "maximum_harm_delta": max(harm_deltas),
    }


stages = {
    "dynamic_original": "after_content_dynamic",
    "parser_fixed": "parser_fixed_output",
    "parser_fixed_guarded": "final_output",
}

group_rows = []

for group_name, group in df.groupby("evaluation_group"):
    for stage, column in stages.items():
        group_rows.append({
            "evaluation_group": group_name,
            "stage": stage,
            **calculate_metrics(group, column),
        })

group_metrics = pd.DataFrame(group_rows)

synthetic = df[
    df["evaluation_group"] == "synthetic_error"
]

per_error_rows = []

for error_type, group in synthetic.groupby("error_type"):
    for stage, column in stages.items():
        per_error_rows.append({
            "error_type": error_type,
            "stage": stage,
            **calculate_metrics(group, column),
        })

per_error_metrics = pd.DataFrame(per_error_rows)

print("\nGROUP METRICS")
display(group_metrics)

print("\nPER-ERROR METRICS")
display(per_error_metrics)

print("\nGUARD-TRIGGERED EXAMPLES")
display(
    df.loc[
        df["guard_triggered"],
        [
            "row_id",
            "evaluation_group",
            "error_type",
            "corrupted_text",
            "parser_fixed_output",
            "final_output",
            "model_edit_distance",
            "model_edit_ratio",
        ],
    ]
)


# --------------------------------------------------
# 7. Save corrected artifacts
# --------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

predictions_parquet = (
    EVALUATION_DIR
    / f"v1_parser_guard_predictions_{timestamp}.parquet"
)

predictions_csv = (
    EVALUATION_DIR
    / f"v1_parser_guard_predictions_{timestamp}.csv"
)

group_metrics_path = (
    EVALUATION_DIR
    / f"v1_parser_guard_group_metrics_{timestamp}.csv"
)

per_error_metrics_path = (
    EVALUATION_DIR
    / f"v1_parser_guard_per_error_metrics_{timestamp}.csv"
)

df.to_parquet(predictions_parquet, index=False)
df.to_csv(predictions_csv, index=False)
group_metrics.to_csv(group_metrics_path, index=False)
per_error_metrics.to_csv(per_error_metrics_path, index=False)

print("\nSaved:")
print(predictions_parquet)
print(predictions_csv)
print(group_metrics_path)
print(per_error_metrics_path)

# Download the row-level CSV so you can send it here.
files.download(str(predictions_csv))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.5 MB/s eta 0:00:00
Mounted at /content/drive
Loaded: /content/drive/MyDrive/kazakh-gec/evaluation/v1_dynamic_budget_predictions_20260811_144916.parquet
Rows: 400

Parser-changed row IDs: [255]
Expected: [255]


,row_id,corrupted_text,clean_text,after_training,after_content_dynamic,parser_fixed_output
255,255,Жауапты органдарға тіисті тапсырмаларын берді.,Жауапты органдарға тиісті тапсырмаларын берді.,Жауапты органдарға тісті тапсырмаларын берді.,ты органдарға тісті тапсырмаларын берді.,Жауапты органдарға тісті тапсырмаларын берді.



Guard-triggered row IDs: [1, 37, 53, 121, 143, 160, 191, 240, 335, 342]
Expected: [1, 37, 53, 121, 143, 160, 191, 240, 335, 342]

GROUP METRICS


,evaluation_group,stage,examples,exact_percent,cer_percent,improved_percent,harmed_percent,copied_percent,catastrophic_percent,maximum_harm_delta
0,clean_control,dynamic_original,100,93.0,0.1369,0.00,7.00,93.00,0.0,5
1,clean_control,parser_fixed,100,93.0,0.1369,0.00,7.00,93.00,0.0,5
2,clean_control,parser_fixed_guarded,100,95.0,0.0489,0.00,5.00,95.00,0.0,1
3,synthetic_error,dynamic_original,300,64.0,0.5541,65.33,8.00,23.33,0.0,6
4,synthetic_error,parser_fixed,300,64.0,0.5372,65.67,7.67,23.33,0.0,6
5,synthetic_error,parser_fixed_guarded,300,64.0,0.4730,65.67,5.00,26.00,0.0,2



PER-ERROR METRICS


...


GUARD-TRIGGERED EXAMPLES


,row_id,evaluation_group,error_type,corrupted_text,parser_fixed_output,final_output,model_edit_distance,model_edit_ratio
1,1,synthetic_error,missing_space,Артында келіншегі мен жалғыз ұлы жәненемерелер...,Артында келіншегі мен жалғыз ұлы және менемере...,Артында келіншегі мен жалғыз ұлы жәненемерелер...,3,0.052632
37,37,synthetic_error,typo,"Трампытң экономикалық саясаты, бүкіл әлем бойы...","Трампытқан экономикалық саясаты, бүкіл әлем бо...","Трампытң экономикалық саясаты, бүкіл әлем бойы...",3,0.033333
53,53,synthetic_error,missing_space,"Шаруақожалығында кейбір жануарлар, кейбір азық...","Шаруақ жоғалығында кейбір жануарлар, кейбір аз...","Шаруақожалығында кейбір жануарлар, кейбір азық...",3,0.042857
121,121,synthetic_error,typo,"Өтінемін, несиені мүмкінідгінше тезірек жабыңыз.","Өтінемін, несиені мүмкінің жағдынше тезірек жа...","Өтінемін, несиені мүмкінідгінше тезірек жабыңыз.",7,0.134615
143,143,synthetic_error,kazakh_letter_loss,Спорт кешеніниң құрылысы осыдан екі жыл бұрын ...,Спорт кешеніңінің құрылысы осыдан екі жыл бұры...,Спорт кешеніниң құрылысы осыдан екі жыл бұрын ...,3,0.052632
160,160,synthetic_error,missing_space,Тамақтанубарысында суретке түсіп отырған отбасы.,Тамақ тұбарысында суретке түсіп отырған отбасы.,Тамақтанубарысында суретке түсіп отырған отбасы.,4,0.083333
191,191,synthetic_error,typo,«Астана» халықаралық қаржы орталығы инститцуио...,«Астана» халықаралық қаржы орталығы институция...,«Астана» халықаралық қаржы орталығы инститцуио...,5,0.062500
240,240,synthetic_error,hyphen_to_space,Кеден органы белгілеген уақытша әкелу (рұқсат ...,Кеден органы белгілеген уақытша әкелу (рұқсат ...,Кеден органы белгілеген уақытша әкелу (рұқсат ...,4,0.013245
335,335,clean_control,clean_identity,"Сот істi, шағымды, наразылықты қарауға әзірлеу...","Сот істi, шағымды, наразылықты қарауға әзірлеу...","Сот істi, шағымды, наразылықты қарауға әзірлеу...",5,0.021834
342,342,clean_control,clean_identity,"Жуынатын бөлмеде дәретхана, шұңғылша, ванна жә...","Жуынатын бөлмеде дәретхана, шұңғылша, ванна жә...","Жуынатын бөлмеде дәретхана, шұңғылша, ванна жә...",4,0.062500



Saved:
/content/drive/MyDrive/kazakh-gec/evaluation/v1_parser_guard_predictions_20260811_160646.parquet
/content/drive/MyDrive/kazakh-gec/evaluation/v1_parser_guard_predictions_20260811_160646.csv
/content/drive/MyDrive/kazakh-gec/evaluation/v1_parser_guard_group_metrics_20260811_160646.csv
/content/drive/MyDrive/kazakh-gec/evaluation/v1_parser_guard_per_error_metrics_20260811_160646.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# STEP 5 — FREEZE V1.1 BASELINE
# No GPU or model loading required.

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import importlib.metadata
import json
import shutil

from google.colab import drive


# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/kazakh-gec")
EVALUATION_DIR = PROJECT_DIR / "evaluation"

RUN_TAG = "20260811_160646"
BASELINE_NAME = f"v1.1_dynamic_parser_guard_{RUN_TAG}"

ADAPTER_SOURCE = (
    PROJECT_DIR
    / "models"
    / "qwen3-0.6b-kazakh-gec"
    / "final"
)

BASELINE_DIR = (
    PROJECT_DIR
    / "baselines"
    / BASELINE_NAME
)

ARCHIVE_PATH = Path(str(BASELINE_DIR) + ".zip")

evaluation_sources = [
    EVALUATION_DIR
    / f"v1_parser_guard_predictions_{RUN_TAG}.parquet",

    EVALUATION_DIR
    / f"v1_parser_guard_predictions_{RUN_TAG}.csv",

    EVALUATION_DIR
    / f"v1_parser_guard_group_metrics_{RUN_TAG}.csv",

    EVALUATION_DIR
    / f"v1_parser_guard_per_error_metrics_{RUN_TAG}.csv",
]


# --------------------------------------------------
# 2. Safety checks — never overwrite a baseline
# --------------------------------------------------

if not ADAPTER_SOURCE.exists():
    raise FileNotFoundError(
        f"Adapter not found: {ADAPTER_SOURCE}"
    )

missing_files = [
    str(path)
    for path in evaluation_sources
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing evaluation files:\n"
        + "\n".join(missing_files)
    )

if BASELINE_DIR.exists() or ARCHIVE_PATH.exists():
    raise FileExistsError(
        f"Baseline already exists:\n{BASELINE_DIR}\n"
        "Nothing was overwritten."
    )


# --------------------------------------------------
# 3. Create snapshot
# --------------------------------------------------

BASELINE_DIR.mkdir(parents=True)

shutil.copytree(
    ADAPTER_SOURCE,
    BASELINE_DIR / "adapter",
)

snapshot_evaluation_dir = (
    BASELINE_DIR / "evaluation"
)

snapshot_evaluation_dir.mkdir()

for source in evaluation_sources:
    shutil.copy2(
        source,
        snapshot_evaluation_dir / source.name,
    )


# --------------------------------------------------
# 4. Save reproducible pipeline configuration
# --------------------------------------------------

configuration = {
    "baseline_name": BASELINE_NAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "model": {
        "base_model": "Qwen/Qwen3-0.6B",
        "adapter_path": "adapter",
    },

    "generation": {
        "do_sample": False,
        "num_beams": 1,
        "dynamic_margin_tokens": 64,
        "hard_max_new_tokens": 512,
        "context_safety_limit": 1024,
        "thinking_enabled": False,
    },

    "parser": {
        "remove_label_only_with_colon": True,
        "label_pattern": (
            r"^\s*(?:Жауап|Түзетілген\s+мәтін)\s*:\s*"
        ),
    },

    "guardrail": {
        "fallback": "return_input",
        "minimum_character_edits": 3,
        "minimum_normalized_edit_ratio": 0.01,
    },

    "verified_metrics": {
        "clean": {
            "examples": 100,
            "exact_percent": 95.0,
            "cer_percent": 0.0489,
            "harmed_percent": 5.0,
            "maximum_harm_delta": 1,
        },
        "synthetic": {
            "examples": 300,
            "exact_percent": 64.0,
            "cer_percent": 0.4730,
            "improved_percent": 65.67,
            "harmed_percent": 5.0,
            "copied_percent": 26.0,
            "maximum_harm_delta": 2,
        },
    },

    "status": (
        "Frozen v1.1 baseline. "
        "Guardrail remains exploratory until validated "
        "on source-disjoint natural and multi-error data."
    ),
}

config_path = BASELINE_DIR / "baseline_config.json"

config_path.write_text(
    json.dumps(
        configuration,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------
# 5. Save reusable parser and guardrail code
# --------------------------------------------------

pipeline_code = r'''import re
import unicodedata
from rapidfuzz.distance import Levenshtein

LABEL_PATTERN = re.compile(
    r"^\s*(?:Жауап|Түзетілген\s+мәтін)\s*:\s*",
    flags=re.IGNORECASE,
)

def normalize_text(value):
    return unicodedata.normalize(
        "NFC",
        str(value),
    ).strip()

def safe_extract(raw_output):
    text = normalize_text(raw_output)
    return LABEL_PATTERN.sub(
        "",
        text,
        count=1,
    ).strip()

def apply_guardrail(input_text, generated_text):
    source = normalize_text(input_text)
    prediction = safe_extract(generated_text)

    edit_distance = Levenshtein.distance(
        source,
        prediction,
    )

    edit_ratio = edit_distance / max(
        len(source),
        len(prediction),
        1,
    )

    if edit_distance >= 3 and edit_ratio >= 0.01:
        return source

    return prediction
'''

(BASELINE_DIR / "inference_pipeline.py").write_text(
    pipeline_code,
    encoding="utf-8",
)


# --------------------------------------------------
# 6. Record library versions
# --------------------------------------------------

package_names = [
    "torch",
    "transformers",
    "peft",
    "accelerate",
    "pandas",
    "pyarrow",
    "rapidfuzz",
]

versions = {}

for package_name in package_names:
    try:
        versions[package_name] = (
            importlib.metadata.version(package_name)
        )
    except importlib.metadata.PackageNotFoundError:
        versions[package_name] = "not installed"

(BASELINE_DIR / "environment_versions.json").write_text(
    json.dumps(
        versions,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------
# 7. README
# --------------------------------------------------

readme = f"""# {BASELINE_NAME}

Frozen Kazakh GEC v1.1 baseline.

Contents:
- adapter/: trained PEFT adapter and tokenizer files
- evaluation/: final row-level predictions and metrics
- baseline_config.json: generation and guardrail settings
- inference_pipeline.py: parser and copy-backoff logic
- environment_versions.json: installed package versions
- manifest.sha256: integrity checksums

Verified result:
- Clean exact: 95%
- Synthetic exact: 64%
- Synthetic CER: 0.4730%
- Synthetic harmed: 5%
- Catastrophic outputs: 0

Important:
Do not train, edit, or overwrite files in this folder.
The guardrail must still be evaluated on a new source-disjoint,
natural, multi-error validation set.
"""

(BASELINE_DIR / "README.md").write_text(
    readme,
    encoding="utf-8",
)


# --------------------------------------------------
# 8. Generate SHA-256 integrity manifest
# --------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


manifest_files = sorted(
    path
    for path in BASELINE_DIR.rglob("*")
    if path.is_file()
    and path.name != "manifest.sha256"
)

manifest_lines = []

for path in manifest_files:
    relative_path = path.relative_to(BASELINE_DIR)

    manifest_lines.append(
        f"{sha256_file(path)}  {relative_path}"
    )

manifest_path = BASELINE_DIR / "manifest.sha256"

manifest_path.write_text(
    "\n".join(manifest_lines) + "\n",
    encoding="utf-8",
)


# --------------------------------------------------
# 9. Verify every checksum
# --------------------------------------------------

for line in manifest_path.read_text(
    encoding="utf-8"
).splitlines():

    expected_hash, relative_path = line.split(
        "  ",
        maxsplit=1,
    )

    actual_hash = sha256_file(
        BASELINE_DIR / relative_path
    )

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Checksum failed: {relative_path}"
        )


# --------------------------------------------------
# 10. Create a single backup archive
# --------------------------------------------------

archive_created = shutil.make_archive(
    str(BASELINE_DIR),
    "zip",
    root_dir=BASELINE_DIR,
)

total_bytes = sum(
    path.stat().st_size
    for path in BASELINE_DIR.rglob("*")
    if path.is_file()
)

print("\nBASELINE VERIFIED AND FROZEN")
print("Folder:", BASELINE_DIR)
print("Archive:", archive_created)
print("Manifest entries:", len(manifest_files))
print(
    "Snapshot size:",
    round(total_bytes / (1024 ** 2), 2),
    "MB",
)


BASELINE VERIFIED AND FROZEN
Folder: /content/drive/MyDrive/kazakh-gec/baselines/v1.1_dynamic_parser_guard_20260811_160646
Archive: /content/drive/MyDrive/kazakh-gec/baselines/v1.1_dynamic_parser_guard_20260811_160646.zip
Manifest entries: 14
Snapshot size: 50.94 MB
